# A Classical (Non-LLM) Wordle Solver

**Purpose.** Establish a rigorous *symbolic + information-theoretic* baseline for Wordle,
to be compared later against a 0.5B LLM trained with SFT/GRPO. The research question this
notebook exists to answer is:

> **How much of Wordle performance is achievable with explicit constraint solving and
> information theory, with no learned model at all?**

Nothing here is AI, machine learning, or reinforcement learning. There is no model being
trained, no gradient, no policy, and no language model. Every choice is a deterministic
computation over an enumerable hypothesis space. The word "solver" is used in its exact
sense: a search procedure over explicit constraints.

## The three layers being measured separately

The notebook is deliberately structured so the contribution of each layer is visible in
isolation, because "the classical baseline" is really three different things:

| Layer | What it does | Knowledge used | Solvers |
|---|---|---|---|
| **1. Symbolic constraint solving** | Deduce exactly which words remain logically possible given the game history. Exact, complete, no approximation. | The rules of Wordle only | *(shared by all)*; `random` isolates it |
| **2. Information-theoretic decision making** | Choose the probe that maximises expected information / minimises posterior size. Uses only the partition structure the feedback function induces. | No English knowledge at all | `entropy`, `expected`, `minimax` |
| **3. Probability / frequency heuristics** | Prefer words whose letters are empirically typical of answers. | Letter statistics of the answer list | `frequency`, and the prior term of `hybrid` |

Reading the benchmark table across these groups is the point of the exercise. `random`
measures what perfect elimination alone buys you; the Layer-2 solvers measure what
information theory adds on top; `frequency` measures whether cheap letter statistics can
substitute for it (spoiler: only partly).

## Structure

1. Project overview · 2. Dataset & environment audit · 3. Imports & configuration ·
4. Load and validate vocabulary · 5. Feedback implementation · 6. Feedback unit tests ·
7. Candidate filtering · 8. Feedback matrix construction · 9. Artifact persistence ·
10.–15. The six solvers · 16. Full benchmark · 17. First-guess analysis ·
18. Example games · 19. Results comparison · 20. Portable artifacts · 21. Local usage

## Portability contract

The core algorithm has no Kaggle dependency. Kaggle path detection happens **only** in the
setup layer of this notebook (Section 2). `wordle_solver.py` reads a single configurable
`DATA_DIR` / `ARTIFACT_DIR`, uses relative paths, and depends on nothing beyond the Python
standard library and NumPy. CPU only — a GPU would provide no benefit here, so none is used.

---
# 2. Dataset & Environment Audit

Nothing about the data is assumed. This section **discovers** whatever word lists are
present, measures them, and reports what was actually found. It searches, in order:

1. `/kaggle/input/**` (any attached Kaggle dataset, arbitrary layout)
2. `./data`, `./input`, `.` (local runs)
3. If nothing usable is found, and `ALLOW_DOWNLOAD` is set, it fetches the canonical lists.

The audit answers the nine questions below with measurements, not assumptions.

In [ ]:
# ---------------------------------------------------------------------------
# Setup layer — the ONLY place that knows about Kaggle. Everything downstream
# uses DATA_DIR / ARTIFACT_DIR and nothing else.
# ---------------------------------------------------------------------------
import os, sys, glob, json, platform, collections, unicodedata, hashlib, time

ON_KAGGLE = os.path.isdir("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if ON_KAGGLE:
    SEARCH_DIRS  = sorted(glob.glob("/kaggle/input/*")) + ["/kaggle/input", "."]
    ARTIFACT_DIR = "/kaggle/working/artifacts"
    DATA_DIR     = "/kaggle/working/data"      # only used if we have to download
else:
    SEARCH_DIRS  = ["data", "input", "."]
    ARTIFACT_DIR = "artifacts"
    DATA_DIR     = "data"

# If no dataset is found locally, may we fetch the canonical lists over the network?
ALLOW_DOWNLOAD = True

print(f"environment      : {'KAGGLE' if ON_KAGGLE else 'LOCAL'}")
print(f"python           : {platform.python_version()}")
print(f"platform         : {platform.platform()}")
print(f"search dirs      : {SEARCH_DIRS}")
print(f"DATA_DIR         : {DATA_DIR}")
print(f"ARTIFACT_DIR     : {ARTIFACT_DIR}")

In [ ]:
# ---------------------------------------------------------------------------
# Discovery: find every file that plausibly contains a 5-letter word list, and
# measure it. No filename, schema, or word count is assumed.
# ---------------------------------------------------------------------------
WORD_LEN = 5
EXTS = (".txt", ".csv", ".json", ".tsv", ".dat", ".words", ".parquet")

def extract_words(path, word_len=WORD_LEN, max_bytes=64 * 1024 * 1024):
    """Best-effort extraction of `word_len`-letter alphabetic tokens from any text-ish file."""
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".parquet":
            try:
                import pandas as pd
                df = pd.read_parquet(path)
            except Exception:
                return None, "parquet unreadable"
            toks = [str(v) for c in df.columns for v in df[c].tolist()]
        elif ext == ".json":
            with open(path, encoding="utf-8", errors="replace") as fh:
                obj = json.load(fh)
            toks = []
            def walk(o):
                if isinstance(o, str): toks.append(o)
                elif isinstance(o, dict):
                    for v in o.values(): walk(v)
                elif isinstance(o, (list, tuple)):
                    for v in o: walk(v)
            walk(obj)
        else:
            if os.path.getsize(path) > max_bytes:
                return None, "too large"
            with open(path, encoding="utf-8", errors="replace") as fh:
                raw = fh.read()
            toks = raw.replace(",", "\n").replace("\t", "\n").split()
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"

    words, raw_n = [], len(toks)
    for t in toks:
        w = t.strip().strip('"').strip("'").lower()
        if len(w) == word_len and w.isascii() and w.isalpha():
            words.append(w)
    return words, f"{len(words)}/{raw_n} tokens were {word_len}-letter alpha"

# Directories to skip. ARTIFACT_DIR is excluded because it contains this
# notebook's OWN generated answers.txt / valid_guesses.txt — on a repeat local
# run those would be rediscovered as if they were source data.
SKIP_PARTS = {os.path.normpath(ARTIFACT_DIR), "artifacts", "tests", "__pycache__",
              ".git", ".conda", ".ipynb_checkpoints", "site-packages"}

def skipped(path):
    parts = set(os.path.normpath(path).split(os.sep))
    return bool(parts & SKIP_PARTS) or os.path.normpath(path).startswith(
        os.path.normpath(ARTIFACT_DIR) + os.sep)

# Scan
found = []
seen_paths = set()
for d in SEARCH_DIRS:
    if not os.path.isdir(d):
        continue
    for path in sorted(glob.glob(os.path.join(d, "**", "*"), recursive=True)):
        if not os.path.isfile(path) or path in seen_paths:
            continue
        if not path.lower().endswith(EXTS) or skipped(path):
            continue
        seen_paths.add(path)
        words, note = extract_words(path)
        if words and len(words) >= 100:          # ignore incidental matches
            found.append({
                "path": path, "n_words": len(words), "n_unique": len(set(words)),
                "size_kb": os.path.getsize(path) / 1024, "note": note, "words": words,
            })

print(f"scanned {len(seen_paths)} candidate files; "
      f"{len(found)} contain >=100 five-letter words\n")
if found:
    print(f"{'path':<62}{'words':>8}{'unique':>8}{'KiB':>9}")
    print("-" * 87)
    for f in sorted(found, key=lambda x: -x["n_words"]):
        print(f"{f['path'][-60:]:<62}{f['n_words']:>8}{f['n_unique']:>8}{f['size_kb']:>9.1f}")
else:
    print("No word lists discovered in the search paths.")

In [ ]:
# ---------------------------------------------------------------------------
# Fallback: if discovery found nothing, fetch the canonical lists.
# ---------------------------------------------------------------------------
CANONICAL = {
    "wordle_answers.txt":
        "https://gist.githubusercontent.com/cfreshman/a03ef2cba789d8cf00c08f767e0fad7b/raw/wordle-answers-alphabetical.txt",
    "wordle_allowed_guesses.txt":
        "https://gist.githubusercontent.com/cfreshman/cdcdf777450c5b5301e439061d29694c/raw/wordle-allowed-guesses.txt",
}

if not found and ALLOW_DOWNLOAD:
    import urllib.request
    os.makedirs(DATA_DIR, exist_ok=True)
    print("no dataset discovered -> downloading canonical lists\n")
    for name, url in CANONICAL.items():
        dest = os.path.join(DATA_DIR, name)
        raw = urllib.request.urlopen(url, timeout=60).read()
        with open(dest, "wb") as fh:
            fh.write(raw)
        words, note = extract_words(dest)
        found.append({"path": dest, "n_words": len(words), "n_unique": len(set(words)),
                      "size_kb": len(raw) / 1024, "note": note, "words": words})
        print(f"  {name}: {len(raw)} bytes, {len(words)} words, "
              f"sha256={hashlib.sha256(raw).hexdigest()[:16]}...")
elif not found:
    raise SystemExit(
        "No word lists found and ALLOW_DOWNLOAD is False. Attach a Kaggle dataset "
        "containing a 5-letter answer list and a larger valid-guess list, or set "
        "ALLOW_DOWNLOAD = True."
    )

In [ ]:
# ---------------------------------------------------------------------------
# Classify the discovered files into ANSWERS vs VALID GUESSES.
#
# Two conventions exist in the wild and they are distinguished by measurement,
# not by filename:
#   (a) the guess file lists only NON-ANSWER extras  -> the lists are disjoint
#   (b) the guess file is already the COMPLETE pool   -> answers are a subset
# ---------------------------------------------------------------------------
ANSWER_HINTS = ("answer", "solution", "target", "secret")
GUESS_HINTS  = ("guess", "allowed", "valid", "dictionary", "words", "all")

def classify(found):
    if len(found) == 1:
        f = found[0]
        print("WARNING: only one word list found. Using it as BOTH the answer list "
              "and the guess pool; the answer/guess distinction cannot be tested.")
        return f, f
    scored = []
    for f in found:
        base = os.path.basename(f["path"]).lower()
        a_hit = any(h in base for h in ANSWER_HINTS)
        g_hit = any(h in base for h in GUESS_HINTS)
        scored.append((f, a_hit, g_hit))
    ans = [f for f, a, g in scored if a and not g]
    gue = [f for f, a, g in scored if g and not a]
    # Fall back on size: the guess pool is always the larger list.
    if not ans or not gue:
        by_size = sorted(found, key=lambda x: x["n_unique"])
        ans, gue = [by_size[0]], [by_size[-1]]
        print("filename hints inconclusive -> classifying by size "
              "(smaller = answers, larger = guesses)")
    a = min(ans, key=lambda x: x["n_unique"])
    g = max(gue, key=lambda x: x["n_unique"])
    return a, g

f_ans, f_gue = classify(found)
ANSWERS_PATH, GUESSES_PATH = f_ans["path"], f_gue["path"]

A_set, G_set = set(f_ans["words"]), set(f_gue["words"])
overlap = len(A_set & G_set)
GUESSES_ARE_EXTRA = (overlap == 0)
full_pool = len(A_set | G_set)

print(f"\nANSWERS file : {ANSWERS_PATH}  ({len(A_set)} unique)")
print(f"GUESSES file : {GUESSES_PATH}  ({len(G_set)} unique)")
print(f"overlap      : {overlap}")
print(f"convention   : guess file holds "
      f"{'NON-ANSWER EXTRAS (disjoint lists)' if GUESSES_ARE_EXTRA else 'the COMPLETE pool'}")
print(f"=> full legal guess pool = {full_pool}")

In [ ]:
# ---------------------------------------------------------------------------
# The nine audit questions, answered by measurement.
# ---------------------------------------------------------------------------
def audit_file(path, words):
    with open(path, "rb") as fh:
        raw_b = fh.read()
    raw = raw_b.decode("utf-8", errors="replace")
    toks = [t for t in raw.replace(",", "\n").split() if t]
    lens = collections.Counter(len(t) for t in toks)
    nonaz = sorted({c for t in toks for c in t if not ("a" <= c <= "z")})
    dupes = [w for w, c in collections.Counter(words).items() if c > 1]
    return {
        "path": path,
        "sha256": hashlib.sha256(raw_b).hexdigest(),
        "bytes": len(raw_b),
        "n_words": len(words),
        "n_unique": len(set(words)),
        "duplicates": len(dupes),
        "duplicate_examples": dupes[:5],
        "length_distribution": dict(sorted(lens.items())),
        "all_length_5": set(lens) == {5},
        "all_lowercase": all(t == t.lower() for t in toks),
        "all_ascii": all(t.isascii() for t in toks),
        "all_alpha": all(t.isalpha() for t in toks),
        "chars_outside_a_z": nonaz,
        "n_accented": sum(1 for t in toks if unicodedata.normalize("NFKD", t) != t),
        "crlf_line_endings": raw.count("\r\n"),
        "sorted_ascending": toks == sorted(toks),
        "blank_lines": sum(1 for l in raw.split("\n") if not l.strip()),
    }

AUDIT = {
    "answers": audit_file(ANSWERS_PATH, f_ans["words"]),
    "guesses": audit_file(GUESSES_PATH, f_gue["words"]),
}

for k, a in AUDIT.items():
    print(f"===== {k.upper()}: {a['path']} =====")
    for field in ("bytes", "n_words", "n_unique", "duplicates", "length_distribution",
                  "all_length_5", "all_lowercase", "all_ascii", "all_alpha",
                  "chars_outside_a_z", "n_accented", "crlf_line_endings",
                  "sorted_ascending", "blank_lines"):
        print(f"  {field:<22} {a[field]}")
    print(f"  {'sha256':<22} {a['sha256'][:32]}...")
    print()

In [ ]:
# ---------------------------------------------------------------------------
# Is this the standard NYT-style Wordle vocabulary?
#
# The original (pre-NYT-revision) Wordle shipped 2315 answers and 10657
# additional allowed guesses, i.e. a 12972-word legal pool. Checking against
# those numbers tells us whether the discovered data is the standard vocabulary,
# an NYT-revised variant, or something else entirely.
# ---------------------------------------------------------------------------
STD_ANSWERS, STD_EXTRA, STD_POOL = 2315, 10657, 12972
n_a, n_pool = len(A_set), full_pool

if n_a == STD_ANSWERS and n_pool == STD_POOL:
    verdict = ("EXACT MATCH to the standard original Wordle vocabulary "
               "(2315 answers / 12972 legal guesses).")
elif abs(n_a - STD_ANSWERS) <= 60 and abs(n_pool - STD_POOL) <= 120:
    verdict = (f"CLOSE to standard ({n_a} vs {STD_ANSWERS} answers, "
               f"{n_pool} vs {STD_POOL} pool) — likely an NYT-revised variant, "
               f"which added and removed a small number of words.")
else:
    verdict = (f"NOT the standard Wordle vocabulary ({n_a} answers, {n_pool} pool). "
               f"Results here are not comparable to published Wordle solver numbers.")

# Marker words: the NYT revision removed several slurs/obscurities from the answer list.
markers = {w: (w in A_set) for w in
           ("agora", "pupal", "lynch", "slave", "wench", "darky", "fibre", "bloke")}

print("VOCABULARY IDENTIFICATION")
print(f"  answers in list      : {n_a}   (standard: {STD_ANSWERS})")
print(f"  full legal pool      : {n_pool}   (standard: {STD_POOL})")
print(f"  non-answer extras    : {len(G_set - A_set)}   (standard: {STD_EXTRA})")
print(f"  verdict              : {verdict}")
print(f"\n  marker words present in the answer list: {markers}")

print("\nTRAIN/TEST SPLITS")
print("  None found, and none is appropriate. This is not a learning problem: there are")
print("  no parameters fitted to held-out data. The full answer list IS the evaluation")
print("  set, and every solver is evaluated on all of it. The frequency model does read")
print("  the answer list, so its statistics are in-sample by construction — noted in")
print("  Section 11 rather than papered over with a split.")

print("\nEXISTING WORDLE CODE IN THE WORKSPACE")
hits = [p for p in seen_paths if p.lower().endswith(".py")]
print(f"  python files found in search paths: {len(hits)}")
print("  No pre-existing feedback implementation or evaluation harness was found in the")
print("  discovered data directories; the implementation in Section 5 is written from")
print("  scratch and validated against hand-derived cases in Section 6.")

### Audit conclusions

Fill in from the output above. On the reference run (canonical lists) the findings were:

| # | Question | Finding |
|---|---|---|
| 1 | Official answer words | `wordle_answers.txt` — 2,315 words |
| 2 | Larger valid-guess dictionary | `wordle_allowed_guesses.txt` — 10,657 words (non-answer extras) |
| 3 | Exact counts | 2,315 answers; 10,657 extras; **12,972** total legal guesses |
| 4 | Length / normalization | All exactly 5 chars, all lowercase ASCII `a–z`; no normalization needed |
| 5 | Duplicates | None, within either file |
| 6 | Punctuation / accents / case | None. No non-`a–z` characters, no accents, no uppercase, no blank lines |
| 7 | Train/test splits | None present, and none appropriate — see above |
| 8 | Existing feedback implementation | None found; written from scratch here |
| 9 | Standard NYT-style vocabulary? | Exact match to the original Wordle vocabulary (2,315 / 12,972) |

The two lists are **disjoint**, so the guess file holds non-answer extras and the legal
pool is their union. This distinction matters: restricting probes to the 2,315 answers
would discard most of the available information-seeking power.

**The source datasets are never modified.** Everything written goes to `ARTIFACT_DIR`.

---
# 3. Imports and Configuration

The next two cells write `wordle_solver.py` and `benchmark.py` to the working directory.
This is what makes the notebook self-contained on Kaggle, where those files do not exist.
They are byte-identical to the standalone module you will download and run locally, so the
notebook is never testing something different from what you ship.

> If you have edited `wordle_solver.py` locally, **skip these two cells** — `%%writefile`
> overwrites unconditionally.

In [ ]:
%%writefile wordle_solver.py
"""
wordle_solver.py — a classical (non-LLM, non-RL) Wordle solver.

This module is deliberately self-contained and depends only on the Python
standard library plus NumPy. It contains no machine-learning models, no neural
networks, no reinforcement learning and no language models. Every decision made
by every solver in this file is an explicit symbolic or information-theoretic
computation over an enumerable hypothesis space.

The implementation is organised into three conceptually distinct layers, and the
separation is intentional — it is the thing being measured:

  1. SYMBOLIC CONSTRAINT SOLVING
     `feedback`, `filter_candidates`, `FeedbackMatrix`. Given a game history,
     deduce exactly which words remain logically possible. This layer is
     complete and exact: it never guesses and never approximates.

  2. INFORMATION-THEORETIC DECISION MAKING
     `EntropySolver`, `ExpectedRemainingSolver`, `MinimaxSolver`. Given the
     surviving hypothesis set, choose the probe that maximises expected
     information gain (or minimises expected / worst-case posterior size).
     No knowledge of English is used — only the partition structure induced by
     the feedback function.

  3. PROBABILITY / FREQUENCY HEURISTICS
     `FrequencySolver`, and the answer-probability term of `HybridSolver`.
     Uses empirical letter statistics of the answer list. This is the only
     layer that encodes anything resembling "prior knowledge" about words.

FEEDBACK ENCODING
-----------------
A feedback pattern is 5 tiles, each one of:

    0 = B  grey   (letter not present, or all its occurrences already accounted for)
    1 = Y  yellow (letter present, wrong position)
    2 = G  green  (letter present, correct position)

Patterns are encoded as a single base-3 integer, little-endian by position:

    code = sum(tile[i] * 3**i  for i in range(5))

so `code` lies in [0, 243). Because 243 < 256 the entire guess x answer
feedback table fits in a `uint8` array — 12972 x 2315 is only ~28.6 MiB, which
is what makes exhaustive precomputation practical on Kaggle or a laptop.

The all-green pattern "GGGGG" is code 242.
"""

from __future__ import annotations

import json
import os
import random
import time
from collections import Counter
from dataclasses import dataclass, field, asdict
from typing import Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np

__all__ = [
    "WORD_LEN", "N_PATTERNS", "ALL_GREEN",
    "feedback", "feedback_code", "code_to_pattern", "pattern_to_code",
    "is_consistent", "filter_candidates",
    "Vocabulary", "load_vocabulary",
    "build_feedback_matrix", "FeedbackMatrix",
    "FrequencyModel",
    "SolverConfig", "GameResult",
    "Solver", "RandomSolver", "FrequencySolver", "EntropySolver",
    "ExpectedRemainingSolver", "MinimaxSolver", "HybridSolver",
    "STRATEGIES", "make_solver",
    "play_game", "solve", "interactive_play",
    "save_artifacts", "load_artifacts", "WordleSolverBundle",
]

WORD_LEN = 5
N_PATTERNS = 3 ** WORD_LEN          # 243
ALL_GREEN = N_PATTERNS - 1          # 242, i.e. "GGGGG"
_POW3 = tuple(3 ** i for i in range(WORD_LEN))
_TILE_CHARS = "BYG"


# =============================================================================
# LAYER 1 — SYMBOLIC: the feedback function
# =============================================================================

def feedback(guess: str, answer: str) -> str:
    """Return the Wordle feedback for `guess` against `answer` as e.g. "GYBBB".

    This is the reference implementation: pure Python, no NumPy, no caching.
    Everything else in this module is validated against it.

    Duplicate-letter semantics (this is the part everyone gets wrong):

    Think of the answer as supplying a *budget* of tiles for each letter, equal
    to how many times that letter occurs in the answer. Two passes:

      Pass 1 — greens. Every position where guess and answer agree is green and
               consumes one unit of that letter's budget.
      Pass 2 — yellows, scanned left to right. A non-green guess position is
               yellow only if its letter still has unconsumed budget, and it
               then consumes one unit. Otherwise it is grey.

    Greens therefore always take priority over yellows regardless of position,
    and surplus occurrences in the guess come back grey. Left-to-right ordering
    in pass 2 is what makes the function deterministic when a letter is
    over-guessed.
    """
    if len(guess) != WORD_LEN or len(answer) != WORD_LEN:
        raise ValueError(
            f"both words must be length {WORD_LEN}; got {guess!r} and {answer!r}"
        )

    tiles = [0] * WORD_LEN
    # Pass 1: greens consume from the budget first.
    remaining = Counter(answer)
    for i in range(WORD_LEN):
        if guess[i] == answer[i]:
            tiles[i] = 2
            remaining[guess[i]] -= 1
    # Pass 2: yellows, left to right, from whatever budget survives.
    for i in range(WORD_LEN):
        if tiles[i] == 2:
            continue
        c = guess[i]
        if remaining[c] > 0:
            tiles[i] = 1
            remaining[c] -= 1
    return "".join(_TILE_CHARS[t] for t in tiles)


def feedback_code(guess: str, answer: str) -> int:
    """Same as `feedback` but returns the base-3 integer encoding."""
    return pattern_to_code(feedback(guess, answer))


def pattern_to_code(pattern: str) -> int:
    """Encode a pattern string (chars in B/Y/G, case-insensitive) as an int."""
    p = pattern.strip().upper()
    if len(p) != WORD_LEN:
        raise ValueError(f"pattern must be {WORD_LEN} chars; got {pattern!r}")
    code = 0
    for i, ch in enumerate(p):
        # Accept a few common aliases so interactive input is forgiving.
        if ch in ("B", "_", "0", "."):
            t = 0
        elif ch in ("Y", "1", "?"):
            t = 1
        elif ch in ("G", "2", "+"):
            t = 2
        else:
            raise ValueError(f"bad feedback character {ch!r} in {pattern!r}")
        code += t * _POW3[i]
    return code


def code_to_pattern(code: int) -> str:
    """Decode a base-3 integer back into a pattern string like "GYBBB"."""
    if not 0 <= code < N_PATTERNS:
        raise ValueError(f"code must be in [0,{N_PATTERNS}); got {code}")
    out = []
    for _ in range(WORD_LEN):
        out.append(_TILE_CHARS[code % 3])
        code //= 3
    return "".join(out)


# =============================================================================
# LAYER 1 — SYMBOLIC: candidate filtering
# =============================================================================

def is_consistent(candidate: str, guess: str, pattern: str) -> bool:
    """True iff `candidate` could still be the answer given (guess -> pattern).

    Defined *by* the feedback function rather than by re-deriving constraint
    rules, which is what guarantees the two can never disagree: a word survives
    exactly when it would have produced the observed pattern.
    """
    return feedback(guess, candidate) == pattern.strip().upper()


def filter_candidates(
    candidates: Iterable[str],
    guess: str,
    pattern: str,
) -> List[str]:
    """Eliminate every candidate inconsistent with a single (guess, pattern).

    Applying this once per history entry is equivalent to filtering against the
    complete history, because consistency with each observation is independent
    of the others (each is a pure function of the candidate). `filter_history`
    does that fold for convenience.
    """
    target = pattern.strip().upper()
    return [w for w in candidates if feedback(guess, w) == target]


def filter_history(
    candidates: Iterable[str],
    history: Sequence[Tuple[str, str]],
) -> List[str]:
    """Filter against a whole game history: [(guess, pattern), ...]."""
    out = list(candidates)
    for guess, pattern in history:
        out = filter_candidates(out, guess, pattern)
    return out


# =============================================================================
# Vocabulary loading / validation
# =============================================================================

@dataclass
class Vocabulary:
    """The two word lists the game is played over.

    `answers`   — words that can actually be the hidden solution.
    `guesses`   — every word the game will *accept* as a guess. This is a
                  superset of `answers`; keeping them distinct is essential,
                  because restricting probes to `answers` throws away most of
                  the available information-seeking power.
    """

    answers: List[str]
    guesses: List[str]
    source: Dict[str, str] = field(default_factory=dict)

    def __post_init__(self) -> None:
        self.answer_index = {w: i for i, w in enumerate(self.answers)}
        self.guess_index = {w: i for i, w in enumerate(self.guesses)}
        # Every answer must be a legal guess, or the game is inconsistent.
        missing = [w for w in self.answers if w not in self.guess_index]
        if missing:
            raise ValueError(
                f"{len(missing)} answers are not legal guesses, e.g. {missing[:5]}"
            )

    @property
    def n_answers(self) -> int:
        return len(self.answers)

    @property
    def n_guesses(self) -> int:
        return len(self.guesses)

    def answer_positions(self) -> np.ndarray:
        """Indices, within `guesses`, of the words that are possible answers."""
        return np.array([self.guess_index[w] for w in self.answers], dtype=np.int32)

    def describe(self) -> str:
        return (
            f"Vocabulary(answers={self.n_answers}, legal_guesses={self.n_guesses}, "
            f"non_answer_guesses={self.n_guesses - self.n_answers})"
        )


def normalize_words(
    raw: str,
    word_len: int = WORD_LEN,
    *,
    strict: bool = False,
) -> Tuple[List[str], Dict[str, int]]:
    """Split raw text into normalized words and report what had to be cleaned.

    Normalization: strip whitespace, lowercase, drop empties. Words that are the
    wrong length or contain anything outside a-z are rejected (and counted)
    rather than silently mangled. Duplicates are removed, first occurrence wins,
    original order preserved.
    """
    import unicodedata

    stats = {
        "raw_tokens": 0, "empty": 0, "uppercased": 0, "accented": 0,
        "bad_length": 0, "non_alpha": 0, "duplicates": 0, "kept": 0,
    }
    words: List[str] = []
    seen = set()
    # Split on any whitespace and also on commas, so simple CSV columns work.
    for tok in raw.replace(",", "\n").split():
        stats["raw_tokens"] += 1
        w = tok.strip().strip('"').strip("'")
        if not w:
            stats["empty"] += 1
            continue
        if w != w.lower():
            stats["uppercased"] += 1
            w = w.lower()
        decomposed = unicodedata.normalize("NFKD", w)
        if decomposed != w:
            stats["accented"] += 1
            w = "".join(c for c in decomposed if not unicodedata.combining(c))
        if len(w) != word_len:
            stats["bad_length"] += 1
            continue
        if not (w.isascii() and w.isalpha()):
            stats["non_alpha"] += 1
            continue
        if w in seen:
            stats["duplicates"] += 1
            continue
        seen.add(w)
        words.append(w)
    stats["kept"] = len(words)
    if strict and (stats["bad_length"] or stats["non_alpha"]):
        raise ValueError(f"rejected words during normalization: {stats}")
    return words, stats


def load_vocabulary(
    answers_path: str,
    guesses_path: Optional[str] = None,
    *,
    guesses_are_extra: Optional[bool] = None,
) -> Vocabulary:
    """Load a `Vocabulary` from two text files (one word per line).

    `guesses_are_extra` controls how the guess file is interpreted:
      True   — the file lists only *non-answer* extras; the legal pool is
               answers + extras.
      False  — the file is already the complete legal pool.
      None   — infer it: if the two lists are disjoint, treat the guess file as
               extras; otherwise assume it is already complete.
    This inference exists because both conventions are common in the wild and
    guessing wrong changes the size of the guess pool.
    """
    with open(answers_path, encoding="utf-8") as fh:
        answers, _ = normalize_words(fh.read())
    if guesses_path is None:
        guesses = list(answers)
    else:
        with open(guesses_path, encoding="utf-8") as fh:
            extra, _ = normalize_words(fh.read())
        aset = set(answers)
        if guesses_are_extra is None:
            overlap = len(aset & set(extra))
            guesses_are_extra = (overlap == 0)
        if guesses_are_extra:
            guesses = list(answers) + [w for w in extra if w not in aset]
        else:
            guesses = list(extra)
            missing = [w for w in answers if w not in set(guesses)]
            guesses.extend(missing)
    guesses = sorted(set(guesses))
    return Vocabulary(
        answers=sorted(answers),
        guesses=guesses,
        source={"answers_path": str(answers_path), "guesses_path": str(guesses_path)},
    )


# =============================================================================
# LAYER 1 — SYMBOLIC: precomputed feedback matrix
# =============================================================================

def _letters_to_idx(words: Sequence[str]) -> np.ndarray:
    """(n, WORD_LEN) uint8 array of 0..25 letter indices."""
    arr = np.frombuffer("".join(words).encode("ascii"), dtype=np.uint8)
    return (arr - ord("a")).reshape(len(words), WORD_LEN)


def build_feedback_matrix(
    guesses: Sequence[str],
    answers: Sequence[str],
    *,
    chunk: int = 512,
    out: Optional[np.ndarray] = None,
    progress: Optional[Callable[[int, int], None]] = None,
) -> np.ndarray:
    """Compute the full feedback table, shape (len(guesses), len(answers)), uint8.

    `M[g, a] == feedback_code(guesses[g], answers[a])`.

    Vectorized restatement of the two-pass rule. Greens are a plain positional
    equality test. Yellows are handled one letter of the alphabet at a time:

      * `ng[g, a, i]`  — position i of the guess holds letter c and the answer
                         does *not* hold c there, i.e. a non-green occurrence.
      * `remaining`    — occurrences of c in the answer minus greens of c, which
                         is exactly the yellow budget after pass 1.
      * `rank`         — exclusive prefix count of non-green occurrences of c to
                         the left of i. The left-to-right greedy rule is then
                         simply `rank < remaining`.

    Chunking over guesses bounds peak memory at roughly
    `chunk * n_answers * WORD_LEN` bytes for the boolean temporaries.
    """
    ng_, na_ = len(guesses), len(answers)
    G = _letters_to_idx(guesses)
    A = _letters_to_idx(answers)

    if out is None:
        out = np.empty((ng_, na_), dtype=np.uint8)
    elif out.shape != (ng_, na_) or out.dtype != np.uint8:
        raise ValueError(f"`out` must be uint8 with shape {(ng_, na_)}")

    pow3 = np.array(_POW3, dtype=np.int16)
    green_w = (2 * pow3).astype(np.int16)

    # Per-letter answer masks and counts, computed once.
    a_eq = [(A == c) for c in range(26)]                       # each (na, 5) bool
    a_cnt = np.stack([m.sum(axis=1) for m in a_eq]).astype(np.int16)   # (26, na)

    for start in range(0, ng_, chunk):
        gl = G[start:start + chunk]
        c_len = gl.shape[0]

        # --- pass 1: greens -------------------------------------------------
        green = (gl[:, None, :] == A[None, :, :])              # (c, na, 5)
        codes = np.tensordot(green.astype(np.int16), green_w, axes=([2], [0]))

        # --- pass 2: yellows, per alphabet letter ---------------------------
        present = np.unique(gl)   # only letters actually used in this chunk
        for c in present:
            gmask = (gl == c)                                  # (c, 5)
            am = a_eq[c]                                       # (na, 5)
            gm3 = gmask[:, None, :]
            # non-green occurrences of c in the guess
            ng_mask = gm3 & ~am[None, :, :]                    # (c, na, 5)
            # greens of c consume budget first
            g_cnt = (gm3 & am[None, :, :]).sum(axis=2, dtype=np.int16)
            budget = a_cnt[c][None, :] - g_cnt                 # (c, na)
            rank = np.cumsum(ng_mask, axis=2, dtype=np.int16) - ng_mask
            yellow = ng_mask & (rank < budget[:, :, None])
            codes += np.tensordot(yellow.astype(np.int16), pow3, axes=([2], [0]))

        out[start:start + c_len] = codes.astype(np.uint8)
        if progress is not None:
            progress(min(start + chunk, ng_), ng_)
    return out


class FeedbackMatrix:
    """Thin wrapper tying a precomputed matrix to its vocabulary.

    Holds the (n_guesses, n_answers) uint8 table plus the index of each answer
    inside the guess list, so a solver can talk about "the guess pool" and "the
    surviving answers" without repeatedly rebuilding lookups.
    """

    def __init__(self, matrix: np.ndarray, vocab: Vocabulary):
        if matrix.shape != (vocab.n_guesses, vocab.n_answers):
            raise ValueError(
                f"matrix shape {matrix.shape} does not match vocabulary "
                f"{(vocab.n_guesses, vocab.n_answers)}"
            )
        if matrix.dtype != np.uint8:
            raise ValueError(f"matrix must be uint8, got {matrix.dtype}")
        self.M = matrix
        self.vocab = vocab
        self.answer_pos = vocab.answer_positions()

    # -- construction ------------------------------------------------------
    @classmethod
    def build(
        cls,
        vocab: Vocabulary,
        *,
        chunk: int = 512,
        progress: Optional[Callable[[int, int], None]] = None,
    ) -> "FeedbackMatrix":
        M = build_feedback_matrix(
            vocab.guesses, vocab.answers, chunk=chunk, progress=progress
        )
        return cls(M, vocab)

    @property
    def nbytes(self) -> int:
        return int(self.M.size)

    # -- symbolic filtering, but as array ops ------------------------------
    def filter_indices(
        self,
        cand_idx: np.ndarray,
        guess: str,
        code: int,
    ) -> np.ndarray:
        """Restrict candidate answer indices to those consistent with (guess, code).

        Exactly `filter_candidates`, expressed as a row lookup. Verified against
        the pure-Python path in the test suite.
        """
        gi = self.vocab.guess_index[guess]
        row = self.M[gi]
        return cand_idx[row[cand_idx] == code]

    def partition_stats(
        self,
        guess_idx: np.ndarray,
        cand_idx: np.ndarray,
        *,
        chunk: int = 2048,
    ) -> Dict[str, np.ndarray]:
        """Partition statistics for every guess in `guess_idx`.

        For guess g, the candidate set is partitioned by observed feedback. Let
        `k` be the bucket sizes and `n = len(cand_idx)`. Returns, per guess:

          entropy   H(g) = -sum p log2 p,  with p = k / n
          expected  E[remaining] = sum k * (k/n) = sum k^2 / n
          worst     max k

        PROBABILITY MODEL — stated explicitly because it is the one modelling
        choice here: every surviving candidate is treated as equally likely,
        p = 1/n. This is the correct posterior under a uniform prior over the
        answer list, which is what the real game does (the daily answer is drawn
        from the answer list, not weighted by English word frequency). So
        p(feedback) is just bucket_size / n. No word-frequency prior is used by
        the information-theoretic solvers; that lives only in `FrequencyModel`.
        """
        n = int(len(cand_idx))
        m = int(len(guess_idx))
        ent = np.empty(m, dtype=np.float64)
        exp = np.empty(m, dtype=np.float64)
        worst = np.empty(m, dtype=np.int32)

        for s in range(0, m, chunk):
            gi = guess_idx[s:s + chunk]
            sub = self.M[np.ix_(gi, cand_idx)].astype(np.int32)   # (b, n)
            b = sub.shape[0]
            # Row-wise histogram over the 243 possible patterns via one bincount.
            flat = sub + (np.arange(b, dtype=np.int32) * N_PATTERNS)[:, None]
            hist = np.bincount(
                flat.ravel(), minlength=b * N_PATTERNS
            ).reshape(b, N_PATTERNS).astype(np.float64)

            p = hist / n
            # log2 only where p > 0; empty buckets contribute 0 to the entropy.
            logp = np.zeros_like(p)
            np.log2(p, out=logp, where=(p > 0))
            ent[s:s + b] = -(p * logp).sum(axis=1)
            exp[s:s + b] = (hist * hist).sum(axis=1) / n
            worst[s:s + b] = hist.max(axis=1).astype(np.int32)
        return {"entropy": ent, "expected_remaining": exp, "worst_case": worst}


# =============================================================================
# LAYER 3 — PROBABILITY / FREQUENCY HEURISTICS
# =============================================================================

@dataclass
class FrequencyModel:
    """Empirical letter statistics of the answer list.

    Two tables:
      `letter_prob[c]`         — fraction of answers containing letter c at all
                                 (presence, not multiplicity).
      `positional_prob[i][c]`  — fraction of answers with letter c at position i.

    Deliberately *presence*-based rather than count-based: scoring a word by
    summing per-occurrence letter frequencies rewards words like "eerie" for
    repeating a common letter, even though a repeat conveys strictly less new
    information than a fresh letter would. See `FrequencySolver.score` for how
    this is used.
    """

    letter_prob: Dict[str, float]
    positional_prob: List[Dict[str, float]]
    n_answers: int
    positional_weight: float = 0.5

    @classmethod
    def fit(cls, answers: Sequence[str], positional_weight: float = 0.5) -> "FrequencyModel":
        n = len(answers)
        pres = Counter()
        for w in answers:
            pres.update(set(w))          # set() => presence, not multiplicity
        pos = [Counter() for _ in range(WORD_LEN)]
        for w in answers:
            for i, ch in enumerate(w):
                pos[i][ch] += 1
        letters = [chr(ord("a") + k) for k in range(26)]
        return cls(
            letter_prob={c: pres[c] / n for c in letters},
            positional_prob=[{c: p[c] / n for c in letters} for p in pos],
            n_answers=n,
            positional_weight=positional_weight,
        )

    def score(self, word: str) -> float:
        """Heuristic desirability of `word` as a probe.

        score = sum over the word's DISTINCT letters of presence-probability
              + w * sum over positions of positional-probability

        The first term is the coverage payoff and counts each letter once, so a
        repeated letter earns nothing extra. The second term rewards putting
        common letters where they are commonly found, and is the only term that
        sees position.
        """
        cov = sum(self.letter_prob.get(c, 0.0) for c in set(word))
        pos = sum(self.positional_prob[i].get(word[i], 0.0) for i in range(WORD_LEN))
        return cov + self.positional_weight * pos

    def score_many(self, words: Sequence[str]) -> np.ndarray:
        return np.array([self.score(w) for w in words], dtype=np.float64)

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict) -> "FrequencyModel":
        return cls(
            letter_prob=d["letter_prob"],
            positional_prob=d["positional_prob"],
            n_answers=d["n_answers"],
            positional_weight=d.get("positional_weight", 0.5),
        )


# =============================================================================
# Configuration
# =============================================================================

@dataclass
class SolverConfig:
    """Single place to change every knob that affects solver behaviour."""

    # -- game rules ---------------------------------------------------------
    max_guesses: int = 6

    # -- strategy selection -------------------------------------------------
    strategy: str = "hybrid"

    # -- reproducibility ----------------------------------------------------
    seed: int = 20260817

    # -- guess pool policy --------------------------------------------------
    # "full"        : always score all legal guesses (strongest, slowest)
    # "adaptive"    : full pool while len(candidates) > adaptive_threshold,
    #                 candidates only below it
    # "candidates"  : only ever guess a surviving candidate (fastest, weakest)
    guess_pool: str = "full"
    adaptive_threshold: int = 2

    # Once this few candidates remain, guessing a non-candidate can never help:
    # with <= 2 left, probing cannot beat simply naming one of them.
    endgame_candidates: int = 2

    # Reserve the last guess for an actual candidate — a probe on the final turn
    # is guaranteed to lose.
    force_candidate_on_last_guess: bool = True

    # -- fixed opener -------------------------------------------------------
    # Turn 1 has no information, so its choice is the same for every game. Set
    # this to skip recomputing it 2315 times during a benchmark.
    opening_guess: Optional[str] = None

    # -- hybrid weights (all terms are in BITS; see HybridSolver) -----------
    entropy_weight: float = 1.0        # coefficient on H(guess)
    minimax_weight: float = 0.25       # lambda: penalty on log2(worst case)
    expected_weight: float = 0.25      # nu:     penalty on log2(E[remaining])
    answer_bonus_weight: float = 1.0   # mu:     credit for a possible-answer probe

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict) -> "SolverConfig":
        known = {k: v for k, v in d.items() if k in cls.__dataclass_fields__}
        return cls(**known)


# =============================================================================
# Solvers
# =============================================================================

@dataclass
class GameResult:
    """Outcome of a single game."""

    answer: str
    guesses: List[str]
    patterns: List[str]
    candidates_remaining: List[int]   # after each guess
    solved: bool
    n_guesses: int
    elapsed_s: float = 0.0

    @property
    def score(self) -> int:
        """Guesses used, or max_guesses + 1 as a conventional failure marker."""
        return self.n_guesses if self.solved else len(self.guesses) + 1


class Solver:
    """Base class. Subclasses implement `choose`.

    A solver is a pure function of (surviving candidates, turn number). The
    driver `play_game` owns the constraint bookkeeping so that every strategy
    sees exactly the same, provably-correct hypothesis set — differences between
    strategies are therefore purely differences in *decision rule*.
    """

    name = "base"
    deterministic = True

    def __init__(self, fb: FeedbackMatrix, config: Optional[SolverConfig] = None):
        self.fb = fb
        self.vocab = fb.vocab
        self.config = config or SolverConfig()
        self.rng = random.Random(self.config.seed)
        self._all_guesses = np.arange(self.vocab.n_guesses, dtype=np.int32)
        self._opener_cache: Optional[str] = None

    def reset(self, seed: Optional[int] = None) -> None:
        self.rng = random.Random(self.config.seed if seed is None else seed)

    # -- guess pool policy --------------------------------------------------
    def _pool(self, cand_idx: np.ndarray, turn: int) -> np.ndarray:
        """Which guesses this solver is allowed to score on this turn."""
        cfg = self.config
        n = len(cand_idx)
        cand_as_guess = self.fb.answer_pos[cand_idx]

        # Final turn, or few enough candidates that probing cannot pay off.
        if cfg.force_candidate_on_last_guess and turn >= cfg.max_guesses:
            return cand_as_guess
        if n <= cfg.endgame_candidates:
            return cand_as_guess

        if cfg.guess_pool == "candidates":
            return cand_as_guess
        if cfg.guess_pool == "adaptive" and n <= cfg.adaptive_threshold:
            return cand_as_guess
        return self._all_guesses

    def choose(self, cand_idx: np.ndarray, turn: int) -> str:
        raise NotImplementedError

    # -- opener caching -----------------------------------------------------
    def opening_guess(self) -> str:
        """The turn-1 guess. Identical across games, so computed at most once."""
        if self.config.opening_guess:
            return self.config.opening_guess
        if self._opener_cache is None:
            all_answers = np.arange(self.vocab.n_answers, dtype=np.int32)
            self._opener_cache = self.choose(all_answers, 1)
        return self._opener_cache


class RandomSolver(Solver):
    """Baseline: name a surviving candidate uniformly at random.

    Pure constraint solving with no decision-making at all — it measures how
    much of the score comes from exact elimination alone. Never probes with a
    non-candidate, so it is the "symbolic layer only" reference point.
    """

    name = "random"
    deterministic = False

    def choose(self, cand_idx: np.ndarray, turn: int) -> str:
        i = self.rng.randrange(len(cand_idx))
        return self.vocab.answers[int(cand_idx[i])]

    def opening_guess(self) -> str:
        # Random has no canonical opener; draw one like any other turn.
        if self.config.opening_guess:
            return self.config.opening_guess
        return self.choose(np.arange(self.vocab.n_answers, dtype=np.int32), 1)


class FrequencySolver(Solver):
    """Rank surviving candidates by empirical letter frequency (Layer 3 only).

    Uses no partition structure whatsoever — it never asks what a guess would
    *reveal*, only whether its letters look typical of an answer. Guesses are
    restricted to surviving candidates, which makes it a direct upgrade over
    `RandomSolver` and isolates the value of frequency priors.
    """

    name = "frequency"

    def __init__(self, fb, config=None, model: Optional[FrequencyModel] = None):
        super().__init__(fb, config)
        self.model = model or FrequencyModel.fit(self.vocab.answers)
        self._cand_scores = self.model.score_many(self.vocab.answers)

    def choose(self, cand_idx: np.ndarray, turn: int) -> str:
        scores = self._cand_scores[cand_idx]
        best = int(cand_idx[int(np.argmax(scores))])
        return self.vocab.answers[best]


class _PartitionSolver(Solver):
    """Shared machinery for the three information-theoretic rules."""

    metric = "entropy"
    maximize = True

    def choose(self, cand_idx: np.ndarray, turn: int) -> str:
        n = len(cand_idx)
        if n == 1:
            return self.vocab.answers[int(cand_idx[0])]

        pool = self._pool(cand_idx, turn)
        stats = self.fb.partition_stats(pool, cand_idx)
        vals = stats[self.metric].astype(np.float64)

        # Tie-break toward guesses that could themselves be the answer: among
        # equally informative probes, one that might win outright is strictly
        # better. Without this, solvers waste turns on non-answers.
        cand_set = set(int(x) for x in self.fb.answer_pos[cand_idx])
        is_cand = np.fromiter(
            (int(g) in cand_set for g in pool), dtype=bool, count=len(pool)
        )
        eps = 1e-9
        adj = vals + (eps if self.maximize else -eps) * is_cand

        k = int(np.argmax(adj) if self.maximize else np.argmin(adj))
        return self.vocab.guesses[int(pool[k])]


class EntropySolver(_PartitionSolver):
    """Maximize expected information gain, H(guess) = -sum p log2 p.

    p(feedback) = bucket_size / n_candidates under a uniform posterior over the
    surviving answers (see `FeedbackMatrix.partition_stats`). H is measured in
    bits: it is the expected reduction in log2 of the hypothesis-space size, so
    a guess scoring 5.9 bits is expected to cut ~2315 candidates to ~40.

    This is one-step-greedy — it optimises immediate information, not the true
    minimum-expected-guesses game tree.
    """

    name = "entropy"
    metric = "entropy"
    maximize = True


class ExpectedRemainingSolver(_PartitionSolver):
    """Minimize E[remaining candidates] = sum bucket^2 / n.

    Closely related to entropy but not identical: entropy penalises by log of
    bucket size, this penalises linearly, so it is more tolerant of one large
    bucket if the rest are tiny. Optimises a different moment of the same
    partition.
    """

    name = "expected"
    metric = "expected_remaining"
    maximize = False


class MinimaxSolver(_PartitionSolver):
    """Minimize the largest resulting partition — worst-case, not average.

    Makes no probabilistic assumption at all beyond the candidate set itself:
    it asks only "if an adversary picked the answer, how bad could this get?"
    Tends to have a better maximum guess count and a slightly worse mean than
    the entropy rule.
    """

    name = "minimax"
    metric = "worst_case"
    maximize = False


class HybridSolver(Solver):
    """Combine all three partition statistics plus the answer-probability prior.

    NORMALIZATION — the reason the weights are meaningful rather than arbitrary:
    the four quantities have incompatible units (bits; a candidate count in
    [1, n]; a count in [1, n]; a probability). Rather than z-scoring, which makes
    weights depend on the spread of whatever pool happens to be scored this
    turn, every term is converted into BITS, which are all comparable and stable
    across turns:

        H(g)                         already bits of expected information
        log2(worst_case(g))          bits of hypothesis space left, worst case
        log2(E[remaining](g))        bits of hypothesis space left, in expectation
        p_answer(g) = 1/n if g is a surviving candidate else 0
                                     converted to bits of value as
                                     log2(1 + p_answer), the expected credit for
                                     possibly winning this turn outright

    score = a*H(g) - lam*log2(worst_case) - nu*log2(E[remaining])
            + mu*log2(1 + p_answer(g))

    All four terms are then on a bits scale, so lam = nu = 0.25 genuinely means
    "weight worst-case and expected-size at a quarter of raw information gain",
    and the default mu = 1.0 gives a possible-answer probe a small edge that
    grows as the candidate set shrinks (which is exactly when winning outright
    matters most). Defaults were chosen to be interpretable, then checked
    against the benchmark rather than tuned to it.
    """

    name = "hybrid"

    def __init__(self, fb, config=None, model: Optional[FrequencyModel] = None):
        super().__init__(fb, config)
        self.model = model or FrequencyModel.fit(self.vocab.answers)

    def choose(self, cand_idx: np.ndarray, turn: int) -> str:
        n = len(cand_idx)
        if n == 1:
            return self.vocab.answers[int(cand_idx[0])]

        pool = self._pool(cand_idx, turn)
        stats = self.fb.partition_stats(pool, cand_idx)
        cfg = self.config

        H = stats["entropy"]
        worst = np.log2(np.maximum(stats["worst_case"], 1).astype(np.float64))
        exp = np.log2(np.maximum(stats["expected_remaining"], 1.0))

        cand_set = set(int(x) for x in self.fb.answer_pos[cand_idx])
        is_cand = np.fromiter(
            (int(g) in cand_set for g in pool), dtype=bool, count=len(pool)
        )
        p_answer = is_cand.astype(np.float64) / n
        bonus = np.log2(1.0 + p_answer)

        score = (
            cfg.entropy_weight * H
            - cfg.minimax_weight * worst
            - cfg.expected_weight * exp
            + cfg.answer_bonus_weight * bonus
        )
        k = int(np.argmax(score))
        return self.vocab.guesses[int(pool[k])]


STRATEGIES: Dict[str, type] = {
    "random": RandomSolver,
    "frequency": FrequencySolver,
    "entropy": EntropySolver,
    "expected": ExpectedRemainingSolver,
    "minimax": MinimaxSolver,
    "hybrid": HybridSolver,
}


def make_solver(
    strategy: str,
    fb: FeedbackMatrix,
    config: Optional[SolverConfig] = None,
    model: Optional[FrequencyModel] = None,
) -> Solver:
    """Instantiate a solver by name."""
    key = strategy.lower()
    if key not in STRATEGIES:
        raise KeyError(f"unknown strategy {strategy!r}; choose from {sorted(STRATEGIES)}")
    cls = STRATEGIES[key]
    if cls in (FrequencySolver, HybridSolver):
        return cls(fb, config, model)
    return cls(fb, config)


# =============================================================================
# Game driver
# =============================================================================

def play_game(
    solver: Solver,
    answer: str,
    *,
    max_guesses: Optional[int] = None,
    verbose: bool = False,
    first_guess: Optional[str] = None,
) -> GameResult:
    """Play one full game and return a `GameResult`.

    The driver — not the solver — maintains the candidate set, using the
    precomputed matrix for elimination. Every strategy is therefore evaluated
    against an identical, exact constraint solver.
    """
    fb = solver.fb
    vocab = fb.vocab
    cfg = solver.config
    limit = cfg.max_guesses if max_guesses is None else max_guesses

    if answer not in vocab.answer_index:
        raise ValueError(f"{answer!r} is not in the answer list")

    cand_idx = np.arange(vocab.n_answers, dtype=np.int32)
    guesses: List[str] = []
    patterns: List[str] = []
    remaining: List[int] = []
    t0 = time.perf_counter()
    solved = False

    for turn in range(1, limit + 1):
        if turn == 1:
            g = first_guess or solver.opening_guess()
        else:
            g = solver.choose(cand_idx, turn)

        code = feedback_code(g, answer)
        pat = code_to_pattern(code)
        cand_idx = fb.filter_indices(cand_idx, g, code)

        guesses.append(g)
        patterns.append(pat)
        remaining.append(int(len(cand_idx)))

        if verbose:
            print(f"Guess {turn}: {g.upper()}")
            print(f"Feedback: {pat}")
            print(f"Candidates remaining: {len(cand_idx)}"
                  + (f"  -> {[vocab.answers[i] for i in cand_idx[:8]]}"
                     f"{' ...' if len(cand_idx) > 8 else ''}"
                     if 0 < len(cand_idx) <= 30 else ""))
            print()

        if code == ALL_GREEN:
            solved = True
            break
        if len(cand_idx) == 0:
            # Only reachable if the answer is outside the answer list, or the
            # feedback function and matrix disagree. Both are bugs, not states.
            raise RuntimeError(
                f"candidate set emptied while solving {answer!r} — "
                f"history={list(zip(guesses, patterns))}"
            )

    elapsed = time.perf_counter() - t0
    if verbose:
        if solved:
            print(f"Solved in {len(guesses)} guesses. ({elapsed*1000:.0f} ms)")
        else:
            print(f"FAILED in {limit} guesses. Answer was {answer.upper()}.")

    return GameResult(
        answer=answer, guesses=guesses, patterns=patterns,
        candidates_remaining=remaining, solved=solved,
        n_guesses=len(guesses), elapsed_s=elapsed,
    )


def solve(
    answer: str = "crane",
    *,
    bundle: Optional["WordleSolverBundle"] = None,
    strategy: Optional[str] = None,
    verbose: bool = True,
    artifact_dir: str = "artifacts",
) -> GameResult:
    """Convenience entry point: `solve(answer="CRANE", verbose=True)`.

    Loads artifacts from `artifact_dir` on first use if no bundle is supplied.
    """
    b = bundle or load_artifacts(artifact_dir)
    s = b.solver(strategy)
    return play_game(s, answer.strip().lower(), verbose=verbose)


def interactive_play(
    bundle: Optional["WordleSolverBundle"] = None,
    *,
    strategy: Optional[str] = None,
    artifact_dir: str = "artifacts",
    input_fn: Callable[[str], str] = input,
    print_fn: Callable[..., None] = print,
) -> Optional[str]:
    """Manual play: the solver proposes, you type the feedback you saw.

    Enter feedback as 5 characters using G/Y/B (grey may also be typed as `_`,
    `.` or `0`). Type `q` to quit. Useful for driving the real NYT game.
    """
    b = bundle or load_artifacts(artifact_dir)
    s = b.solver(strategy)
    vocab = b.vocab
    cand_idx = np.arange(vocab.n_answers, dtype=np.int32)

    print_fn(f"Interactive Wordle solver — strategy={s.name}")
    print_fn("Enter feedback as 5 chars of G/Y/B (or 'q' to quit).\n")

    for turn in range(1, s.config.max_guesses + 1):
        g = s.opening_guess() if turn == 1 else s.choose(cand_idx, turn)
        print_fn(f"Guess {turn}: {g.upper()}   ({len(cand_idx)} candidates)")
        while True:
            raw = input_fn("  feedback> ").strip()
            if raw.lower() in ("q", "quit", "exit"):
                print_fn("Aborted.")
                return None
            try:
                code = pattern_to_code(raw)
                break
            except ValueError as exc:
                print_fn(f"  {exc}")
        if code == ALL_GREEN:
            print_fn(f"\nSolved in {turn} guesses: {g.upper()}")
            return g
        cand_idx = b.fb.filter_indices(cand_idx, g, code)
        if len(cand_idx) == 0:
            print_fn("\nNo candidates left — the feedback entered is inconsistent, "
                     "or the answer is not in this answer list.")
            return None
        if len(cand_idx) <= 15:
            print_fn("  remaining: "
                     + ", ".join(vocab.answers[i] for i in cand_idx))
    print_fn("\nOut of guesses.")
    return None


# =============================================================================
# Artifact persistence
# =============================================================================

ARTIFACT_FILES = {
    "answers": "answers.txt",
    "guesses": "valid_guesses.txt",
    "matrix": "feedback_matrix.npy",
    "metadata": "metadata.json",
    "frequency": "frequency_model.json",
    "config": "solver_config.json",
}


def _portable_path(p: Optional[str]) -> Optional[str]:
    """Reduce a path to something machine-agnostic for recording in metadata.

    Absolute paths from the build machine are useless to anyone else and leak the
    builder's directory layout, so they are stored relative to the working
    directory (forward-slashed) and reduced to a bare filename when a relative
    path would be meaningless (different drive, or far outside the tree).
    """
    if not p:
        return p
    p = str(p)
    try:
        rel = os.path.relpath(p, start=os.getcwd()).replace("\\", "/")
    except ValueError:          # different drive on Windows
        return os.path.basename(p)
    return os.path.basename(p) if rel.startswith("../../") else rel


def _is_memmap_of(arr: np.ndarray, path: str) -> bool:
    """True if `arr` is a numpy memmap backed by the file at `path`."""
    if not isinstance(arr, np.memmap):
        return False
    src = getattr(arr, "filename", None)
    if not src:
        return False
    try:
        return os.path.exists(path) and os.path.samefile(src, path)
    except OSError:
        return os.path.abspath(str(src)) == os.path.abspath(path)


def _matrix_unchanged_on_disk(path: str, M: np.ndarray) -> bool:
    """True if `path` already holds exactly the array `M`.

    Lets `save_artifacts` skip a pointless 28 MiB rewrite. That matters for more
    than speed: on Windows, any live memory map of the destination file — held by
    a *different* array, e.g. a bundle reloaded earlier in the same session —
    locks it, and np.save would fail with OSError(EINVAL). Comparing first turns
    the common build -> save -> reload -> save-again flow into a no-op instead of
    a crash.
    """
    if not os.path.exists(path):
        return False
    existing = None
    try:
        existing = np.load(path, mmap_mode="r")
        if existing.shape != M.shape or existing.dtype != M.dtype:
            return False
        return bool(np.array_equal(np.asarray(existing), np.asarray(M)))
    except Exception:
        return False
    finally:
        # Release the mapping promptly, or we become the thing holding the lock.
        mm = getattr(existing, "_mmap", None) if existing is not None else None
        del existing
        if mm is not None:
            try:
                mm.close()
            except Exception:
                pass


def save_artifacts(
    artifact_dir: str,
    vocab: Vocabulary,
    fb: FeedbackMatrix,
    model: FrequencyModel,
    config: SolverConfig,
    *,
    extra_metadata: Optional[dict] = None,
) -> Dict[str, str]:
    """Write a complete, portable solver bundle to `artifact_dir`.

    Everything needed to reconstruct the solver on another machine with only
    Python + NumPy. Paths are relative by design; nothing Kaggle-specific is
    recorded inside the artifacts.
    """
    os.makedirs(artifact_dir, exist_ok=True)
    p = {k: os.path.join(artifact_dir, v) for k, v in ARTIFACT_FILES.items()}

    with open(p["answers"], "w", encoding="utf-8", newline="\n") as fh:
        fh.write("\n".join(vocab.answers) + "\n")
    with open(p["guesses"], "w", encoding="utf-8", newline="\n") as fh:
        fh.write("\n".join(vocab.guesses) + "\n")

    # Skip rewriting the matrix when the destination already holds exactly these
    # bytes — either because `fb.M` maps that very file, or because an identical
    # matrix is already there. Besides saving a 28 MiB write, this is what makes
    # build -> save -> reload -> save-again work on Windows, where any live
    # memory map of the file (possibly held by another array entirely) locks it
    # and np.save would fail with OSError(EINVAL).
    if _is_memmap_of(fb.M, p["matrix"]) or _matrix_unchanged_on_disk(p["matrix"], fb.M):
        pass
    else:
        try:
            np.save(p["matrix"], fb.M)
        except OSError as exc:
            raise OSError(
                f"could not write {p['matrix']!r}: {exc}. If another array is "
                f"memory-mapping this file, load it with mmap=False before saving."
            ) from exc

    with open(p["frequency"], "w", encoding="utf-8") as fh:
        json.dump(model.to_dict(), fh, indent=2)
    with open(p["config"], "w", encoding="utf-8") as fh:
        json.dump(config.to_dict(), fh, indent=2)

    import platform
    meta = {
        "format_version": 1,
        "word_len": WORD_LEN,
        "n_patterns": N_PATTERNS,
        "all_green_code": ALL_GREEN,
        "feedback_encoding": "base3 little-endian per position; 0=B(grey) 1=Y(yellow) 2=G(green); code=sum(tile[i]*3**i)",
        "n_answers": vocab.n_answers,
        "n_guesses": vocab.n_guesses,
        "n_non_answer_guesses": vocab.n_guesses - vocab.n_answers,
        "matrix_shape": list(fb.M.shape),
        "matrix_dtype": str(fb.M.dtype),
        "matrix_bytes": int(fb.M.nbytes),
        "matrix_orientation": "M[guess_index, answer_index]; rows follow valid_guesses.txt, cols follow answers.txt",
        # Recorded relative / basename-only so the bundle carries no absolute
        # paths from the build machine.
        "vocab_source": {k: _portable_path(v) for k, v in vocab.source.items()},
        "built_with": {
            "python": platform.python_version(),
            "numpy": np.__version__,
            "platform": platform.platform(),
        },
        "built_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "strategies": sorted(STRATEGIES),
        "notes": "Classical solver artifacts. No ML/LLM/RL components.",
    }
    if extra_metadata:
        meta.update(extra_metadata)
    with open(p["metadata"], "w", encoding="utf-8") as fh:
        json.dump(meta, fh, indent=2)
    return p


@dataclass
class WordleSolverBundle:
    """A loaded solver: vocabulary + matrix + frequency model + config."""

    vocab: Vocabulary
    fb: FeedbackMatrix
    model: FrequencyModel
    config: SolverConfig
    metadata: dict = field(default_factory=dict)

    def solver(self, strategy: Optional[str] = None) -> Solver:
        return make_solver(strategy or self.config.strategy, self.fb, self.config, self.model)

    def describe(self) -> str:
        return (
            f"{self.vocab.describe()}\n"
            f"matrix {self.fb.M.shape} {self.fb.M.dtype} "
            f"({self.fb.M.nbytes/2**20:.1f} MiB)\n"
            f"default strategy={self.config.strategy}, "
            f"guess_pool={self.config.guess_pool}, seed={self.config.seed}"
        )


def load_artifacts(
    artifact_dir: str = "artifacts",
    *,
    mmap: bool = True,
) -> WordleSolverBundle:
    """Reload a bundle written by `save_artifacts` — no rebuilding required.

    `mmap=True` memory-maps the feedback matrix, so start-up is instant and the
    ~28 MiB table is paged in on demand rather than copied into RAM.
    """
    p = {k: os.path.join(artifact_dir, v) for k, v in ARTIFACT_FILES.items()}
    for k in ("answers", "guesses", "matrix"):
        if not os.path.exists(p[k]):
            raise FileNotFoundError(
                f"missing artifact {p[k]!r}. Build artifacts first "
                f"(see build_artifacts.py or the notebook)."
            )

    with open(p["answers"], encoding="utf-8") as fh:
        answers = [w for w in fh.read().split() if w]
    with open(p["guesses"], encoding="utf-8") as fh:
        guesses = [w for w in fh.read().split() if w]
    vocab = Vocabulary(answers=answers, guesses=guesses)

    M = np.load(p["matrix"], mmap_mode="r" if mmap else None)
    fb = FeedbackMatrix(M, vocab)

    if os.path.exists(p["frequency"]):
        with open(p["frequency"], encoding="utf-8") as fh:
            model = FrequencyModel.from_dict(json.load(fh))
    else:
        model = FrequencyModel.fit(answers)

    if os.path.exists(p["config"]):
        with open(p["config"], encoding="utf-8") as fh:
            config = SolverConfig.from_dict(json.load(fh))
    else:
        config = SolverConfig()

    metadata = {}
    if os.path.exists(p["metadata"]):
        with open(p["metadata"], encoding="utf-8") as fh:
            metadata = json.load(fh)

    return WordleSolverBundle(vocab=vocab, fb=fb, model=model,
                              config=config, metadata=metadata)


# =============================================================================
# CLI
# =============================================================================

def _main(argv: Optional[List[str]] = None) -> int:
    import argparse

    ap = argparse.ArgumentParser(
        prog="wordle_solver",
        description="Classical Wordle solver (no ML/LLM/RL).",
    )
    ap.add_argument("--artifact-dir", default="artifacts",
                    help="directory holding the built artifacts (default: artifacts)")
    ap.add_argument("--strategy", default=None, choices=sorted(STRATEGIES),
                    help="override the strategy stored in solver_config.json")
    ap.add_argument("--answer", default=None,
                    help="solve this answer verbosely, e.g. --answer crane")
    ap.add_argument("--interactive", action="store_true",
                    help="play against the real game, entering feedback by hand")
    ap.add_argument("--info", action="store_true", help="describe the loaded bundle")
    args = ap.parse_args(argv)

    bundle = load_artifacts(args.artifact_dir)
    if args.info or (not args.answer and not args.interactive):
        print(bundle.describe())
        if not args.answer and not args.interactive:
            return 0
    if args.answer:
        play_game(bundle.solver(args.strategy), args.answer.strip().lower(), verbose=True)
    if args.interactive:
        interactive_play(bundle, strategy=args.strategy)
    return 0


if __name__ == "__main__":
    raise SystemExit(_main())

In [ ]:
%%writefile benchmark.py
"""
benchmark.py — evaluation harness for the classical Wordle solvers.

Kept separate from `wordle_solver.py` so the solver module stays free of
evaluation-only code. Depends on NumPy only; produces plain dicts and lists so
the notebook can turn them into tables or plots without this module knowing
anything about pandas or matplotlib.
"""

from __future__ import annotations

import json
import math
import os
import statistics
import time
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Sequence

import numpy as np

from wordle_solver import (
    FeedbackMatrix, FrequencyModel, GameResult, Solver, SolverConfig,
    make_solver, play_game,
)

__all__ = [
    "BenchmarkResult", "run_benchmark", "summarize", "summary_table",
    "first_guess_analysis", "candidate_trajectory",
]


# =============================================================================
# Running games
# =============================================================================

@dataclass
class BenchmarkResult:
    """All per-game outcomes for one strategy, plus timing."""

    strategy: str
    games: List[GameResult] = field(default_factory=list)
    wall_s: float = 0.0
    config: Dict = field(default_factory=dict)
    n_answers_evaluated: int = 0
    opening_guess: Optional[str] = None

    @property
    def scores(self) -> np.ndarray:
        """Guesses used per game; failures counted as max_guesses + 1."""
        return np.array([g.score for g in self.games], dtype=np.float64)

    @property
    def solved_mask(self) -> np.ndarray:
        return np.array([g.solved for g in self.games], dtype=bool)


def run_benchmark(
    strategy: str,
    fb: FeedbackMatrix,
    config: SolverConfig,
    *,
    answers: Optional[Sequence[str]] = None,
    model: Optional[FrequencyModel] = None,
    progress_every: int = 250,
    log: Optional[Callable[[str], None]] = print,
) -> BenchmarkResult:
    """Play one game per answer and collect results.

    The opening guess is computed once and reused, since turn 1 sees no
    information and is therefore identical across games. For the stochastic
    `random` strategy the opener is redrawn per game from the seeded RNG, so the
    run stays reproducible without being artificially fixed.
    """
    solver = make_solver(strategy, fb, config, model)
    solver.reset()
    pool = list(answers if answers is not None else fb.vocab.answers)

    fixed_opener: Optional[str] = None
    if solver.deterministic:
        t0 = time.perf_counter()
        fixed_opener = solver.opening_guess()
        if log:
            log(f"  [{strategy}] opener = {fixed_opener.upper()} "
                f"({time.perf_counter() - t0:.1f}s)")

    games: List[GameResult] = []
    t_start = time.perf_counter()
    for i, ans in enumerate(pool, 1):
        games.append(play_game(solver, ans, first_guess=fixed_opener))
        if log and progress_every and i % progress_every == 0:
            el = time.perf_counter() - t_start
            mean_so_far = float(np.mean([g.score for g in games]))
            log(f"  [{strategy}] {i}/{len(pool)}  "
                f"mean={mean_so_far:.4f}  {el:.1f}s  "
                f"(eta {el / i * (len(pool) - i):.0f}s)")
    wall = time.perf_counter() - t_start

    return BenchmarkResult(
        strategy=strategy, games=games, wall_s=wall,
        config=config.to_dict(), n_answers_evaluated=len(pool),
        opening_guess=fixed_opener,
    )


# =============================================================================
# Summarizing
# =============================================================================

def summarize(res: BenchmarkResult, max_guesses: int = 6) -> Dict:
    """Descriptive statistics for one strategy.

    `mean_guesses` counts only SOLVED games, which is the number normally quoted
    for Wordle solvers. `mean_score` counts failures as `max_guesses + 1` and is
    the honest single-number comparison when failure rates differ, so both are
    reported.
    """
    scores = res.scores
    solved = res.solved_mask
    n = len(scores)
    solved_scores = scores[solved]

    dist = {k: int(((scores == k) & solved).sum()) for k in range(1, max_guesses + 1)}
    pct = {k: 100.0 * v / n for k, v in dist.items()}

    out = {
        "strategy": res.strategy,
        "n_games": n,
        "opening_guess": res.opening_guess,
        "mean_guesses_solved_only": float(np.mean(solved_scores)) if len(solved_scores) else float("nan"),
        "mean_score_failures_penalized": float(np.mean(scores)),
        "median_guesses": float(np.median(solved_scores)) if len(solved_scores) else float("nan"),
        "std_guesses": float(np.std(solved_scores, ddof=1)) if len(solved_scores) > 1 else 0.0,
        "min_guesses": int(np.min(solved_scores)) if len(solved_scores) else None,
        "max_guesses": int(np.max(solved_scores)) if len(solved_scores) else None,
        "n_solved": int(solved.sum()),
        "n_failed": int((~solved).sum()),
        "solve_rate_pct": 100.0 * float(solved.mean()),
        "failure_rate_pct": 100.0 * float((~solved).mean()),
        "distribution_counts": dist,
        "distribution_pct": pct,
        "failures": [g.answer for g in res.games if not g.solved][:50],
        "wall_s": res.wall_s,
        "ms_per_game": 1000.0 * res.wall_s / n if n else 0.0,
    }
    return out


def candidate_trajectory(res: BenchmarkResult, max_turns: int = 6) -> Dict[int, Dict]:
    """Average surviving candidates after each guess.

    Only games still in progress at a given turn contribute to that turn's
    average — a solved game has left the pool, so including its (absent) later
    turns would bias the number downward. `n_games_reaching_turn` is reported so
    the average can be read in context.
    """
    out: Dict[int, Dict] = {}
    for t in range(1, max_turns + 1):
        vals = [g.candidates_remaining[t - 1]
                for g in res.games if len(g.candidates_remaining) >= t]
        if not vals:
            continue
        out[t] = {
            "n_games_reaching_turn": len(vals),
            "mean_remaining": float(np.mean(vals)),
            "median_remaining": float(np.median(vals)),
            "max_remaining": int(np.max(vals)),
            "mean_bits_remaining": float(np.mean([math.log2(v) for v in vals if v > 0])),
        }
    return out


def summary_table(summaries: Sequence[Dict], max_guesses: int = 6) -> str:
    """Fixed-width comparison table — no pandas needed."""
    hdr = (f"{'strategy':<11}{'open':<7}{'mean':>7}{'med':>5}{'std':>6}"
           f"{'min':>4}{'max':>4}" +
           "".join(f"{'%'+str(k):>7}" for k in range(1, max_guesses + 1)) +
           f"{'fail%':>7}{'ms/game':>9}")
    lines = [hdr, "-" * len(hdr)]
    for s in summaries:
        row = (
            f"{s['strategy']:<11}"
            f"{(s['opening_guess'] or '-')[:6]:<7}"
            f"{s['mean_guesses_solved_only']:>7.4f}"
            f"{s['median_guesses']:>5.0f}"
            f"{s['std_guesses']:>6.3f}"
            f"{s['min_guesses']:>4}"
            f"{s['max_guesses']:>4}"
            + "".join(f"{s['distribution_pct'].get(k, 0.0):>7.2f}"
                      for k in range(1, max_guesses + 1))
            + f"{s['failure_rate_pct']:>7.2f}"
            + f"{s['ms_per_game']:>9.1f}"
        )
        lines.append(row)
    return "\n".join(lines)


# =============================================================================
# First-guess analysis
# =============================================================================

def first_guess_analysis(
    fb: FeedbackMatrix,
    *,
    model: Optional[FrequencyModel] = None,
    chunk: int = 2048,
) -> Dict[str, np.ndarray]:
    """Score every legal guess as an opener against the full answer list.

    Turn 1 is the one decision where all strategies face an identical, fully
    known state (every answer is possible), so it is the cleanest place to see
    what each objective actually optimises for and where they disagree.

    Returns arrays aligned with `fb.vocab.guesses`.
    """
    vocab = fb.vocab
    all_answers = np.arange(vocab.n_answers, dtype=np.int32)
    all_guesses = np.arange(vocab.n_guesses, dtype=np.int32)
    stats = fb.partition_stats(all_guesses, all_answers, chunk=chunk)

    is_answer = np.zeros(vocab.n_guesses, dtype=bool)
    is_answer[fb.answer_pos] = True

    m = model or FrequencyModel.fit(vocab.answers)
    freq = m.score_many(vocab.guesses)

    return {
        "word": np.array(vocab.guesses, dtype=object),
        "entropy": stats["entropy"],
        "expected_remaining": stats["expected_remaining"],
        "worst_case": stats["worst_case"],
        "is_possible_answer": is_answer,
        "frequency_score": freq,
    }


def top_openers(
    fga: Dict[str, np.ndarray],
    metric: str,
    n: int = 20,
    *,
    maximize: bool = True,
) -> List[Dict]:
    """Top-n openers by one metric, each row carrying all the other metrics too."""
    vals = fga[metric]
    order = np.argsort(-vals if maximize else vals, kind="stable")[:n]
    rows = []
    for rank, i in enumerate(order, 1):
        rows.append({
            "rank": rank,
            "word": str(fga["word"][i]),
            "entropy_bits": float(fga["entropy"][i]),
            "expected_remaining": float(fga["expected_remaining"][i]),
            "worst_case_remaining": int(fga["worst_case"][i]),
            "is_possible_answer": bool(fga["is_possible_answer"][i]),
            "frequency_score": float(fga["frequency_score"][i]),
        })
    return rows


def format_openers(rows: Sequence[Dict], title: str) -> str:
    hdr = (f"{'#':>3} {'word':<7}{'H(bits)':>9}{'E[rem]':>10}"
           f"{'worst':>7}{'answer?':>9}{'freq':>7}")
    out = [title, hdr, "-" * len(hdr)]
    for r in rows:
        out.append(
            f"{r['rank']:>3} {r['word']:<7}{r['entropy_bits']:>9.4f}"
            f"{r['expected_remaining']:>10.2f}{r['worst_case_remaining']:>7}"
            f"{('yes' if r['is_possible_answer'] else 'no'):>9}"
            f"{r['frequency_score']:>7.3f}"
        )
    return "\n".join(out)


# =============================================================================
# Persistence of results
# =============================================================================

def save_benchmark(
    path: str,
    summaries: Sequence[Dict],
    trajectories: Dict[str, Dict],
    *,
    extra: Optional[Dict] = None,
) -> str:
    """Write benchmark summaries to JSON (per-game detail is intentionally omitted)."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    payload = {
        "summaries": list(summaries),
        "candidate_trajectories": trajectories,
        "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    if extra:
        payload.update(extra)
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(payload, fh, indent=2)
    return path

## Reproducibility and the single configuration block

Everything that affects behaviour lives in `SolverConfig`. Change it here, nowhere else.

In [ ]:
import importlib, sys, random
import numpy as np

for m in ("wordle_solver", "benchmark"):
    if m in sys.modules:
        importlib.reload(sys.modules[m])

import wordle_solver as ws
import benchmark as bm
importlib.reload(ws); importlib.reload(bm)

from wordle_solver import (
    WORD_LEN, N_PATTERNS, ALL_GREEN,
    feedback, feedback_code, code_to_pattern, pattern_to_code,
    filter_candidates, filter_history, is_consistent,
    Vocabulary, load_vocabulary, FeedbackMatrix, build_feedback_matrix,
    FrequencyModel, SolverConfig, GameResult,
    RandomSolver, FrequencySolver, EntropySolver, ExpectedRemainingSolver,
    MinimaxSolver, HybridSolver, STRATEGIES, make_solver,
    play_game, solve, interactive_play,
    save_artifacts, load_artifacts,
)
from benchmark import (
    run_benchmark, summarize, summary_table, candidate_trajectory,
    first_guess_analysis, top_openers, format_openers, save_benchmark,
)

# =====================  CONFIGURATION  =====================================
SEED            = 20260817   # every stochastic solver derives its RNG from this
MAX_GUESSES     = 6          # standard Wordle
STRATEGY        = "hybrid"   # default strategy stored in solver_config.json

# Guess pool policy for the information-theoretic solvers:
#   "full"       always score all 12972 legal guesses  (strongest, slowest)
#   "adaptive"   full pool until few candidates remain  (much faster)
#   "candidates" only ever guess a surviving candidate  (fastest, weakest)
GUESS_POOL      = "full"
ADAPTIVE_THRESHOLD = 2

# Hybrid weights. All four terms are in BITS, so these are directly comparable
# (see Section 15 for the derivation and why normalization comes first).
ENTROPY_WEIGHT  = 1.00       # coefficient on H(guess)
MINIMAX_WEIGHT  = 0.25       # lambda: penalty on log2(worst-case bucket)
EXPECTED_WEIGHT = 0.25       # nu:     penalty on log2(E[remaining])
ANSWER_BONUS    = 1.00       # mu:     credit for probing a possible answer

# Benchmark scope. 0 = evaluate EVERY answer (what the report uses).
BENCHMARK_LIMIT = 0
BENCHMARK_STRATEGIES = ["random", "frequency", "entropy", "expected", "minimax", "hybrid"]

# Matrix build blocking (memory vs speed). 512 peaks well under 1 GiB.
MATRIX_CHUNK    = 512
# ===========================================================================

CONFIG = SolverConfig(
    max_guesses=MAX_GUESSES, strategy=STRATEGY, seed=SEED,
    guess_pool=GUESS_POOL, adaptive_threshold=ADAPTIVE_THRESHOLD,
    entropy_weight=ENTROPY_WEIGHT, minimax_weight=MINIMAX_WEIGHT,
    expected_weight=EXPECTED_WEIGHT, answer_bonus_weight=ANSWER_BONUS,
)

random.seed(SEED)
np.random.seed(SEED)

print("ENVIRONMENT")
print(f"  python        {platform.python_version()}")
print(f"  numpy         {np.__version__}")
print(f"  platform      {platform.platform()}")
print(f"  on kaggle     {ON_KAGGLE}")
try:
    import pandas as pd;      print(f"  pandas        {pd.__version__}")
except ImportError:           print("  pandas        (not installed - optional)")
try:
    import matplotlib;        print(f"  matplotlib    {matplotlib.__version__}")
except ImportError:           print("  matplotlib    (not installed - optional)")
print("\nNo ML / LLM / RL libraries are imported anywhere in this notebook.")
print("\nCONFIG")
for k, v in CONFIG.to_dict().items():
    print(f"  {k:<32} {v}")
print(f"\n  SEED = {SEED}")

---
# 4. Load and Validate Vocabulary

`load_vocabulary` normalizes both lists, infers the answers/extras convention, and asserts
the one invariant that must hold for the game to be coherent: **every answer must be a
legal guess.**

In [ ]:
VOCAB = load_vocabulary(ANSWERS_PATH, GUESSES_PATH,
                        guesses_are_extra=GUESSES_ARE_EXTRA)

print(VOCAB.describe())
print(f"\nanswers      first 5 {VOCAB.answers[:5]}")
print(f"             last  5 {VOCAB.answers[-5:]}")
print(f"legal guesses first 5 {VOCAB.guesses[:5]}")
print(f"             last  5 {VOCAB.guesses[-5:]}")

# Invariants
assert VOCAB.n_answers == len(set(VOCAB.answers)), "duplicate answers"
assert VOCAB.n_guesses == len(set(VOCAB.guesses)), "duplicate guesses"
assert all(len(w) == WORD_LEN for w in VOCAB.guesses), "non-5-letter guess"
assert all(w.isascii() and w.isalpha() and w.islower() for w in VOCAB.guesses)
assert set(VOCAB.answers) <= set(VOCAB.guesses), "an answer is not a legal guess"
assert VOCAB.n_guesses >= VOCAB.n_answers
print("\nall vocabulary invariants hold.")

# Structure of the answer space — this is what the solvers are searching.
import collections as _c
rep = sum(1 for w in VOCAB.answers if len(set(w)) < WORD_LEN)
print(f"\nanswers containing a repeated letter: {rep} "
      f"({100*rep/VOCAB.n_answers:.1f}%)  <- why duplicate-letter feedback matters")
print(f"most common answer letters: "
      f"{_c.Counter(c for w in VOCAB.answers for c in w).most_common(8)}")
print(f"log2(2315) = {np.log2(VOCAB.n_answers):.3f} bits of initial uncertainty")

---
# 5. The Wordle Feedback Function

`feedback(guess, answer) -> "GYBBB"` where `G`=green, `Y`=yellow, `B`=grey.

## Duplicate letters — the whole difficulty

Think of the answer as supplying a **budget** of tiles for each letter, equal to how many
times that letter occurs. Two passes:

1. **Greens.** Every position where guess and answer agree is green and **consumes one unit**
   of that letter's budget.
2. **Yellows, scanned left to right.** A non-green position is yellow only if its letter still
   has unconsumed budget, and it then consumes one unit. Otherwise grey.

Three consequences that naive implementations get wrong:

- **Greens always win, regardless of position.** A green at index 4 consumes budget before a
  yellow at index 1 can claim it. (`added` vs `dread` → `YYBYG`, not `YYBYY`.)
- **Surplus guessed letters come back grey.** (`speed` vs `abide` → `BBYBY`: the second `e`
  greys out.)
- **Left-to-right ordering makes it deterministic** when a letter is over-guessed.

## Encoding

Patterns are encoded as a base-3 integer, little-endian by position:

$$\text{code} = \sum_{i=0}^{4} \text{tile}_i \cdot 3^i, \qquad \text{tile} \in \{0{=}B,\, 1{=}Y,\, 2{=}G\}$$

so `code` ∈ [0, 243). Because **243 < 256, the entire feedback table fits in `uint8`** — which
is exactly what makes exhaustive precomputation cheap (Section 8). `GGGGG` is code 242.

In [ ]:
print("source of the reference implementation:\n")
import inspect
print(inspect.getsource(ws.feedback))

In [ ]:
# Worked examples, printed so the duplicate-letter rules are visible.
def show(g, a, note=""):
    print(f"  {g.upper()} vs {a.upper()}  ->  {feedback(g, a)}   "
          f"code={feedback_code(g, a):>3}   {note}")

print("simple cases")
show("crane", "crane", "identical")
show("slate", "crane", "a and e already in position")
show("plumb", "crane", "nothing shared")
print("\nduplicate letters")
show("speed", "abide", "surplus guessed 'e' greys out")
show("abide", "speed", "NOT symmetric with the line above")
show("eerie", "eaten", "green 'e' consumes budget before later yellows")
show("llama", "lucid", "green 'l' spends the only 'l'")
show("added", "dread", "green at index 4 consumes before yellow at index 1")
show("array", "rural", "two greens consume from both repeated letters")
show("geese", "sheep", "one 'e' green, one yellow, one grey")
show("mummy", "mamma", "triple letter, all matched")
show("aaaaa", "ababa", "degenerate all-same guess")

print(f"\nencoding: GGGGG={pattern_to_code('GGGGG')}  BBBBB={pattern_to_code('BBBBB')}  "
      f"GBBBB={pattern_to_code('GBBBB')}  BBBBG={pattern_to_code('BBBBG')}")
print(f"ALL_GREEN={ALL_GREEN}, N_PATTERNS={N_PATTERNS}")

---
# 6. Feedback Unit Tests

The feedback function is the foundation — every solver's correctness rests on it, and a
subtle duplicate-letter bug would silently corrupt every benchmark number in this notebook
while still looking plausible. So it is tested three ways:

1. **Hand-derived cases** for each duplicate-letter rule, each with its derivation stated.
2. **Property tests** over thousands of random pairs (invariants that must hold universally).
3. **Cross-validation** of the vectorized matrix against this reference (Section 8).

The full `pytest` suite lives in `tests/test_wordle_solver.py` (61 tests). The cell below
re-runs the essential assertions inline so the notebook is self-verifying.

In [ ]:
# ---------------------------------------------------------------------------
# 6a. Hand-derived duplicate-letter cases.
# ---------------------------------------------------------------------------
CASES = [
    # (guess, answer, expected, derivation)
    ("crane", "crane", "GGGGG", "identical words"),
    ("slate", "crane", "BBGBG", "crane = c r a n e; a@2 and e@4 already aligned"),
    ("plumb", "crane", "BBBBB", "no shared letters"),
    ("crane", "nacre", "YYYYG", "e@4 aligned; other four present but displaced"),
    ("crane", "ranec", "YYYYY", "derangement: all present, none aligned"),
    ("speed", "abide", "BBYBY", "budget e=1: e@2 yellow, e@3 grey; d yellow"),
    ("abide", "speed", "BBBYY", "reversed argument order gives a different pattern"),
    ("eerie", "eaten", "GYBBB", "e@0 green spends 1 of 2 e's; e@1 yellow; e@4 grey"),
    ("llama", "lucid", "GBBBB", "l@0 green spends the only l, so l@1 greys"),
    ("added", "dread", "YYBYG", "d@4 green consumes first; then d@1 yellow, d@2 grey"),
    ("array", "rural", "BYGGB", "r@2,a@3 green; a budget exhausted, r@1 yellow"),
    ("geese", "sheep", "BYGYB", "e@2 green; e@1 yellow, s@3 yellow, e@4 grey"),
    ("mummy", "mamma", "GBGGB", "three greens consume all three m's"),
    ("mamma", "mummy", "GBGGB", "symmetric here only because the m's align identically"),
    ("abide", "eerie", "BBYBG", "answer has three e's; guess's single e is green"),
    ("aaaaa", "ababa", "GBGBG", "all-same guess vs two-letter answer"),
    ("eecee", "ceeee", "YGYGG", "left-to-right: leftmost non-green e takes the budget"),
]

fails = []
for g, a, expect, why in CASES:
    got = feedback(g, a)
    ok = got == expect
    if not ok:
        fails.append((g, a, expect, got, why))
    print(f"  {'PASS' if ok else 'FAIL'}  {g.upper()} vs {a.upper()}  "
          f"got={got} expect={expect}   {why}")
assert not fails, f"hand-derived cases failed: {fails}"
print(f"\nall {len(CASES)} hand-derived cases pass.")

In [ ]:
# ---------------------------------------------------------------------------
# 6b. Property tests over random pairs — universal invariants.
# ---------------------------------------------------------------------------
import collections, math
rng = random.Random(SEED)
N = 20000
words = VOCAB.answers
pairs = [(rng.choice(words), rng.choice(words)) for _ in range(N)]

# (1) A letter can never be marked (G or Y) more times than it occurs in the answer.
for g, a in pairs:
    pat = feedback(g, a)
    ac = collections.Counter(a)
    mk = collections.Counter(ch for ch, t in zip(g, pat) if t in "GY")
    for ch, k in mk.items():
        assert k <= ac[ch], f"over-marked {ch!r} in {g}/{a} -> {pat}"

# (2) Green count == number of aligned positions.
for g, a in pairs:
    pat = feedback(g, a)
    assert pat.count("G") == sum(1 for i in range(WORD_LEN) if g[i] == a[i])

# (3) All-green if and only if guess == answer.
for g, a in pairs:
    assert (feedback(g, a) == "GGGGG") == (g == a)

# (4) Codes always in range, and encode/decode round-trips.
for g, a in pairs:
    c = feedback_code(g, a)
    assert 0 <= c < N_PATTERNS
    assert pattern_to_code(feedback(g, a)) == c
for c in range(N_PATTERNS):
    assert pattern_to_code(code_to_pattern(c)) == c

# (5) Total marked tiles <= number of shared letters counted with multiplicity.
for g, a in pairs[:5000]:
    pat = feedback(g, a)
    shared = sum((collections.Counter(g) & collections.Counter(a)).values())
    assert pat.count("G") + pat.count("Y") == shared, (
        f"{g}/{a} -> {pat}: marked tiles must equal the multiset intersection size")

print(f"all property tests pass over {N} random pairs "
      f"and all {N_PATTERNS} pattern codes.")
print("\ninvariants verified:")
print("  1. per-letter marks never exceed the answer's letter count")
print("  2. green count == positional match count")
print("  3. GGGGG <=> guess == answer")
print("  4. codes in [0,243) and encoding round-trips")
print("  5. G+Y count == |multiset intersection of guess and answer|")

In [ ]:
# ---------------------------------------------------------------------------
# 6c. Run the full pytest suite if it is available (it is, locally).
# ---------------------------------------------------------------------------
import subprocess, os
if os.path.isdir("tests"):
    r = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"],
                       capture_output=True, text=True)
    print(r.stdout[-3000:] or r.stderr[-3000:])
else:
    print("tests/ not present (expected on Kaggle) — the inline checks above cover "
          "the same duplicate-letter and property assertions.")

---
# 7. Candidate Filtering — Layer 1, Symbolic Constraint Solving

This is the part that is *exact*. Maintain `candidate_answers`; after each
(guess, feedback) observation, eliminate every answer that could not have produced it.

The key design decision: consistency is defined **by the feedback function itself** —

```python
def is_consistent(candidate, guess, pattern):
    return feedback(guess, candidate) == pattern
```

rather than by re-deriving constraint rules ("green means letter at position", "grey means
letter absent", …). Re-deriving is where duplicate-letter bugs breed, because grey does *not*
mean "absent" when the letter appears elsewhere. Defining consistency this way makes it
**impossible** for the filter and the feedback function to disagree.

Filtering per-observation is equivalent to filtering against the whole history, since each
check is an independent pure function of the candidate.

### ANSWERS vs GUESSES

`candidate_answers` is always a subset of the **answer** list — those are the only things the
hidden word can be. But the **guess pool** stays the full legal dictionary, because a word
that cannot be the answer can still be the most informative probe. Collapsing these two is
the single most common way a Wordle solver is accidentally weakened.

In [ ]:
# Filtering demo on a real game state.
answer = "crane"
history = []
cands = list(VOCAB.answers)
print(f"start: {len(cands)} candidates  ({np.log2(len(cands)):.2f} bits of uncertainty)\n")

for g in ["soare", "clint", "crane"]:
    pat = feedback(g, answer)
    history.append((g, pat))
    before = len(cands)
    cands = filter_candidates(cands, g, pat)
    print(f"guess {g.upper()} -> {pat}")
    print(f"  candidates {before} -> {len(cands)}   "
          f"({np.log2(before/max(len(cands),1)):.2f} bits gained)")
    if len(cands) <= 12:
        print(f"  remaining: {cands}")
    print()

# Equivalence: incremental filtering == filtering against the full history.
assert filter_history(VOCAB.answers, history) == cands
print("incremental filtering == full-history filtering: OK")

In [ ]:
# ---------------------------------------------------------------------------
# Tests proving the filter is consistent with the feedback function.
# ---------------------------------------------------------------------------
rng = random.Random(SEED + 1)

# (1) The true answer NEVER gets eliminated by correct feedback.
for _ in range(2000):
    a, g = rng.choice(VOCAB.answers), rng.choice(VOCAB.guesses)
    assert a in filter_candidates(VOCAB.answers, g, feedback(g, a))

# (2) Survivors are exactly the words whose feedback matches — nothing more, nothing less.
pool = rng.sample(VOCAB.answers, 500)
for _ in range(200):
    a, g = rng.choice(VOCAB.answers), rng.choice(VOCAB.guesses)
    pat = feedback(g, a)
    surv = set(filter_candidates(pool, g, pat))
    for w in pool:
        assert (w in surv) == (feedback(g, w) == pat), f"inconsistent on {w}"

# (3) The 243 patterns PARTITION the candidate set: bucket sizes sum to n exactly.
for g in rng.sample(VOCAB.guesses, 25):
    total = sum(len(filter_candidates(pool, g, code_to_pattern(c)))
                for c in range(N_PATTERNS))
    assert total == len(pool), f"{g}: buckets summed to {total}, expected {len(pool)}"

# (4) Multi-turn: filtering is order-independent and the answer always survives.
for _ in range(300):
    a = rng.choice(VOCAB.answers)
    gs = [rng.choice(VOCAB.guesses) for _ in range(3)]
    hist = [(g, feedback(g, a)) for g in gs]
    fwd = filter_history(VOCAB.answers, hist)
    rev = filter_history(VOCAB.answers, hist[::-1])
    assert sorted(fwd) == sorted(rev), "filtering is order-dependent"
    assert a in fwd

print("candidate filtering is provably consistent with the feedback function:")
print("  1. the true answer is never eliminated              (2000 trials)")
print("  2. survivors == exact feedback agreement            (200 x 500 checks)")
print("  3. the 243 patterns partition the candidates        (25 guesses)")
print("  4. filtering is order-independent                   (300 x 3-turn histories)")

---
# 8. Precomputing the Feedback Matrix

The vocabulary is small enough to precompute **every** guess × answer feedback:

$$M[g, a] = \text{feedback\_code}(\text{guesses}[g],\ \text{answers}[a])$$

Shape **12,972 × 2,315 = 30,030,180 cells**. Since codes fit in `uint8`, that is **28.6 MiB** —
trivially small for Kaggle or a laptop, and it turns every later filter and entropy
computation into an array lookup instead of millions of Python string comparisons.

## How the vectorization works

Greens are a plain positional equality test. Yellows are the hard part, handled one letter
of the alphabet at a time (26 iterations, each fully vectorized):

- `ng[g,a,i]` — position `i` of the guess holds letter `c` and the answer does **not** hold `c`
  there: a *non-green occurrence*.
- `budget` — occurrences of `c` in the answer minus greens of `c`. This is exactly the yellow
  budget surviving pass 1.
- `rank` — exclusive prefix count of non-green occurrences of `c` to the left of `i`.

The left-to-right greedy rule then collapses to a single comparison: **`yellow = rank < budget`**.

Chunking over guesses bounds peak memory at roughly `chunk × n_answers × 5` bytes per
temporary, so `MATRIX_CHUNK = 512` stays well under 1 GiB.

In [ ]:
print(inspect.getsource(ws.build_feedback_matrix))

In [ ]:
# ---------------------------------------------------------------------------
# Build the matrix (or reuse an existing artifact).
# ---------------------------------------------------------------------------
matrix_path = os.path.join(ARTIFACT_DIR, "feedback_matrix.npy")
REBUILD = not os.path.exists(matrix_path)

if REBUILD:
    print(f"building {VOCAB.n_guesses} x {VOCAB.n_answers} uint8 matrix "
          f"({VOCAB.n_guesses*VOCAB.n_answers/2**20:.1f} MiB) ...")
    t0 = time.perf_counter()
    _last = [0.0]
    def _prog(done, total):
        now = time.perf_counter()
        if now - _last[0] > 3.0 or done == total:
            _last[0] = now
            el = now - t0
            print(f"  {done}/{total}  {el:.1f}s  eta {el/max(done,1)*(total-done):.0f}s")
    FB = FeedbackMatrix.build(VOCAB, chunk=MATRIX_CHUNK, progress=_prog)
    BUILD_SECONDS = time.perf_counter() - t0
    print(f"built in {BUILD_SECONDS:.1f}s")
else:
    print(f"reusing existing {matrix_path}")
    FB = FeedbackMatrix(np.load(matrix_path, mmap_mode="r"), VOCAB)
    BUILD_SECONDS = 0.0

print(f"\nshape  {FB.M.shape}")
print(f"dtype  {FB.M.dtype}")
print(f"bytes  {FB.M.nbytes:,} ({FB.M.nbytes/2**20:.1f} MiB)")
print(f"range  [{FB.M.min()}, {FB.M.max()}]  (must be within [0,{N_PATTERNS-1}])")

In [ ]:
# ---------------------------------------------------------------------------
# Verify the vectorized matrix against the pure-Python reference.
# ---------------------------------------------------------------------------
assert FB.M.shape == (VOCAB.n_guesses, VOCAB.n_answers), "matrix dimensions wrong"
assert FB.M.dtype == np.uint8
assert FB.M.max() < N_PATTERNS

# (1) Random spot-check against the reference implementation.
rs = np.random.default_rng(SEED)
NCHK = 50000
gi = rs.integers(0, VOCAB.n_guesses, NCHK)
ai = rs.integers(0, VOCAB.n_answers, NCHK)
bad = [(VOCAB.guesses[g], VOCAB.answers[a], int(FB.M[g, a]), feedback_code(VOCAB.guesses[g], VOCAB.answers[a]))
       for g, a in zip(gi, ai) if FB.M[g, a] != feedback_code(VOCAB.guesses[g], VOCAB.answers[a])]
assert not bad, f"matrix disagrees with reference: {bad[:5]}"
print(f"(1) {NCHK} random cells match the reference implementation exactly")

# (2) Global invariant: the diagonal (answer guessed against itself) is all-green.
diag = np.asarray(FB.M[FB.answer_pos, np.arange(VOCAB.n_answers)])
assert (diag == ALL_GREEN).all(), "diagonal is not all-green"
print(f"(2) all {VOCAB.n_answers} diagonal cells are GGGGG ({ALL_GREEN})")

# (3) Exhaustive check of a few complete rows (every answer for one guess).
for g in ["soare", "crane", "eerie", "mummy", "aaaaa" if "aaaaa" in VOCAB.guess_index else "arise"]:
    if g not in VOCAB.guess_index:
        continue
    row = np.asarray(FB.M[VOCAB.guess_index[g]])
    ref = np.array([feedback_code(g, a) for a in VOCAB.answers], dtype=np.uint8)
    assert np.array_equal(row, ref), f"row mismatch for {g}"
print("(3) complete rows verified exhaustively for several guesses "
      "(including repeated-letter ones)")

# (4) Matrix filtering == pure-Python filtering.
idx0 = np.arange(VOCAB.n_answers, dtype=np.int32)
for _ in range(300):
    a, g = rng.choice(VOCAB.answers), rng.choice(VOCAB.guesses)
    c = feedback_code(g, a)
    got = sorted(VOCAB.answers[i] for i in FB.filter_indices(idx0, g, c))
    want = sorted(filter_candidates(VOCAB.answers, g, code_to_pattern(c)))
    assert got == want
print("(4) matrix-based filtering == pure-Python filtering (300 trials)")

# (5) Every row must partition the answers into buckets summing to n.
for g in rng.sample(VOCAB.guesses, 200):
    row = np.asarray(FB.M[VOCAB.guess_index[g]])
    assert np.bincount(row, minlength=N_PATTERNS).sum() == VOCAB.n_answers
print("(5) every checked row partitions the answer set correctly")
print("\nFEEDBACK MATRIX VERIFIED.")

---
# 9. Artifact Persistence and Reloading

The pipeline is **build → save → reload → run without rebuilding**. The matrix is stored as
a portable `.npy` and reloaded with `mmap_mode="r"`, so start-up is instant and the 28.6 MiB
table pages in on demand rather than being copied into RAM.

Nothing Kaggle-specific is written into the artifacts — no absolute input paths, no
environment assumptions. The bundle is the portability boundary.

In [ ]:
FREQ_MODEL = FrequencyModel.fit(VOCAB.answers)

paths = save_artifacts(
    ARTIFACT_DIR, VOCAB, FB, FREQ_MODEL, CONFIG,
    extra_metadata={
        "matrix_build_seconds": round(BUILD_SECONDS, 2),
        "matrix_build_chunk": MATRIX_CHUNK,
        "discovered_answers_path": ANSWERS_PATH,
        "discovered_guesses_path": GUESSES_PATH,
        "guesses_file_holds_extras_only": bool(GUESSES_ARE_EXTRA),
        "audit": {k: {kk: vv for kk, vv in v.items() if kk != "duplicate_examples"}
                  for k, v in AUDIT.items()},
        "vocabulary_verdict": verdict,
    },
)
print(f"wrote artifacts to {ARTIFACT_DIR}:")
for k, p in paths.items():
    print(f"  {k:<10} {os.path.basename(p):<24} {os.path.getsize(p)/1024:>10.1f} KiB")

In [ ]:
# ---------------------------------------------------------------------------
# Reload round-trip: prove the bundle alone is sufficient.
# ---------------------------------------------------------------------------
t0 = time.perf_counter()
BUNDLE = load_artifacts(ARTIFACT_DIR, mmap=True)
print(f"reloaded in {(time.perf_counter()-t0)*1000:.0f} ms (memory-mapped)\n")
print(BUNDLE.describe())

assert BUNDLE.vocab.answers == VOCAB.answers
assert BUNDLE.vocab.guesses == VOCAB.guesses
assert np.array_equal(np.asarray(BUNDLE.fb.M), np.asarray(FB.M))
assert BUNDLE.config.seed == SEED
assert BUNDLE.model.letter_prob == FREQ_MODEL.letter_prob
assert isinstance(BUNDLE.fb.M, np.memmap), "expected a memory-mapped array"

# A reloaded bundle must play byte-identical games.
_a = "crane" if "crane" in VOCAB.answer_index else VOCAB.answers[0]
r1 = play_game(make_solver("entropy", FB, CONFIG, FREQ_MODEL), _a)
r2 = play_game(BUNDLE.solver("entropy"), _a)
assert r1.guesses == r2.guesses and r1.patterns == r2.patterns
print(f"\nartifact round-trip verified; reloaded solver reproduces the same game "
      f"({' -> '.join(g.upper() for g in r2.guesses)})")

---
# 10. Random Surviving-Answer Baseline

**Layer 1 only.** Filter exactly, then name a surviving candidate uniformly at random. No
decision-making of any kind.

This is the most important reference point in the notebook: it measures **how much of Wordle
is solved by perfect constraint propagation alone**, before any information theory or word
knowledge is applied. Any cleverer strategy must be judged by how far it improves on this.

In [ ]:
RESULTS, SUMMARIES, TRAJECTORIES = {}, {}, {}

def demo(strategy, answers=("crane", "eerie", "mummy"), n_sample=200):
    """Show one verbose game, then a quick sampled score for orientation."""
    solver = make_solver(strategy, FB, CONFIG, FREQ_MODEL)
    print(f"--- {strategy}: example game ---")
    play_game(solver, answers[0], verbose=True)
    rs = np.random.default_rng(SEED)
    samp = [VOCAB.answers[i] for i in rs.choice(VOCAB.n_answers, n_sample, replace=False)]
    r = run_benchmark(strategy, FB, CONFIG, answers=samp, model=FREQ_MODEL, log=None)
    print(f"sampled {n_sample} answers: mean={r.scores.mean():.4f}  "
          f"solved={r.solved_mask.sum()}/{n_sample}  "
          f"{r.wall_s/n_sample*1000:.1f} ms/game")
    return r

_ = demo("random")

---
# 11. Frequency-Based Solver

**Layer 3 only.** Rank surviving candidates by empirical letter statistics of the answer list.
It never asks what a guess would *reveal* — only whether its letters look typical of an answer.

$$\text{score}(w) = \underbrace{\sum_{c \in \text{set}(w)} P(\text{letter } c \text{ present})}_{\text{coverage}} \;+\; w_{\text{pos}} \sum_{i=0}^{4} P(\text{letter } w_i \text{ at position } i)$$

## Handling repeated letters

The coverage term sums over **distinct** letters (`set(w)`), not over positions. This matters:
summing per-occurrence frequencies would reward `eerie` for containing `e` three times, even
though a repeated letter conveys strictly *less* new information than a fresh letter. Counting
each letter once removes that artificial bonus. The positional term is the only one that sees
position, and it is down-weighted (`positional_weight = 0.5`) so it refines rather than
dominates the ranking.

> **In-sample note.** The frequency model is fitted on the same answer list it is evaluated
> against, so its statistics are in-sample by construction. There is no held-out split
> because there are no learned parameters to overfit in the usual sense — but this solver's
> numbers should be read as a mild upper bound on what letter statistics alone can do.

In [ ]:
m = FREQ_MODEL
top = sorted(m.letter_prob.items(), key=lambda kv: -kv[1])[:10]
print("P(letter present in a random answer) — top 10")
for c, p in top:
    print(f"  {c}  {p*100:5.1f}%  {'#'*int(p*100)}")

print("\nmost likely letter at each position")
for i, tbl in enumerate(m.positional_prob):
    best = sorted(tbl.items(), key=lambda kv: -kv[1])[:5]
    print(f"  pos {i}: " + "  ".join(f"{c}={p*100:.1f}%" for c, p in best))

print("\nrepeated letters earn no coverage bonus:")
for w in ["arose", "eerie", "raise", "mummy", "adieu"]:
    cov = sum(m.letter_prob[c] for c in set(w))
    print(f"  {w.upper()}  score={m.score(w):.4f}  "
          f"coverage={cov:.4f} over {len(set(w))} distinct letters")
assert m.score("arose") > m.score("eerie")
print("\n  -> AROSE outscores EERIE despite similar letter frequencies. Correct.")

_ = demo("frequency")

---
# 12. Entropy / Information-Gain Solver

**Layer 2.** For every legal guess, partition the surviving candidates by the feedback the
guess would produce, then choose the guess maximising expected information gain:

$$H(g) = -\sum_{f \in \text{patterns}} p(f) \log_2 p(f), \qquad p(f) = \frac{|\{a \in C : \text{feedback}(g,a) = f\}|}{|C|}$$

## How the probabilities are estimated — stated exactly

Every surviving candidate is treated as **equally likely**: $p(a) = 1/|C|$. So $p(f)$ is just
`bucket_size / n_candidates`.

This is not a modelling shortcut, it is the correct posterior: the real game draws its answer
from the answer list, *not* weighted by English word frequency. A uniform prior over the answer
list, updated by exact constraint filtering, gives a uniform posterior over the survivors. No
word-frequency prior enters here — that lives only in the Layer-3 frequency model.

$H$ is in **bits**: the expected reduction in $\log_2$ of the hypothesis-space size. A guess
scoring 5.9 bits is expected to cut ~2,315 candidates down to ~40.

**This is one-step greedy.** It maximises *immediate* information, not the true
minimum-expected-guesses game tree. Full lookahead would require searching the decision tree
over sequences of guesses; that is a deliberate scope boundary, and it is the main gap between
this baseline and a theoretically optimal solver.

In [ ]:
# Entropy of a few candidate openers, computed from the full answer set.
all_ans = np.arange(VOCAB.n_answers, dtype=np.int32)
probe = ["soare", "roate", "raise", "arise", "crane", "slate", "adieu", "tares", "xylyl"]
probe = [w for w in probe if w in VOCAB.guess_index]
pidx = np.array([VOCAB.guess_index[w] for w in probe], dtype=np.int32)
st = FB.partition_stats(pidx, all_ans)

print(f"scored against all {VOCAB.n_answers} answers "
      f"({np.log2(VOCAB.n_answers):.2f} bits of uncertainty)\n")
print(f"{'word':<8}{'H(bits)':>9}{'E[rem]':>10}{'worst':>7}{'buckets':>9}{'answer?':>9}")
print("-" * 52)
for w, i in zip(probe, range(len(probe))):
    row = np.asarray(FB.M[VOCAB.guess_index[w]])
    nb = int((np.bincount(row, minlength=N_PATTERNS) > 0).sum())
    print(f"{w:<8}{st['entropy'][i]:>9.4f}{st['expected_remaining'][i]:>10.2f}"
          f"{st['worst_case'][i]:>7}{nb:>9}"
          f"{('yes' if w in VOCAB.answer_index else 'no'):>9}")

print("\nnote XYLYL: a legal guess with very low entropy — it splits almost nothing.")
print("This is why the guess pool must be SCORED, not merely allowed.")

_ = demo("entropy")

---
# 13. Expected-Remaining-Candidates Solver

**Layer 2.** Minimise the expected size of the surviving candidate set:

$$\mathbb{E}[\text{remaining}](g) = \sum_f p(f)\cdot|f| = \frac{1}{|C|}\sum_f |f|^2$$

Closely related to entropy but **not** identical, and the difference is instructive: entropy
penalises a bucket by $\log$ of its size, this penalises **linearly**. So this objective is
more tolerant of one large bucket when the rest are tiny, while entropy spreads mass more
evenly. They optimise different moments of the same partition and therefore pick different
openers — `ROATE` versus `SOARE` on this vocabulary.

In [ ]:
_ = demo("expected")

---
# 14. Minimax Solver

**Layer 2, worst-case.** For each guess, take the size of the largest resulting partition and
minimise it:

$$\text{minimax}(g) = \min_g \max_f |\{a \in C : \text{feedback}(g,a) = f\}|$$

This makes **no probabilistic assumption at all** beyond the candidate set itself — it asks
only "if an adversary chose the answer, how bad could this get?" Consequently it typically
achieves a better *maximum* guess count than the entropy rule while giving up a little on the
*mean*. That mean-versus-tail tradeoff is exactly what the benchmark's max column and
distribution histogram are there to expose.

In [ ]:
_ = demo("minimax")

---
# 15. Hybrid Solver

Combines all three partition statistics with the answer-probability prior.

## Normalization comes first — this is the point

The four quantities have **incompatible units**: entropy is in bits; worst-case is a candidate
count in $[1, n]$; expected-remaining is also a count but on a different scale; answer
probability is in $[0,1]$. Adding them with raw weights would make those weights meaningless —
a "weight of 0.25" on a quantity ranging over 2,315 is not comparable to 0.25 on a quantity
ranging over 11 bits.

Two options were considered:

- **z-scoring** each term across the scored pool. Rejected: the mean and spread change every
  turn as the candidate set shrinks, so a fixed weight would silently mean something different
  on turn 2 than on turn 4.
- **Converting everything to bits.** Chosen: bits are stable across turns and directly
  interpretable, and the conversion is the natural one — the $\log_2$ of a hypothesis-space
  size *is* its size in bits.

So every term becomes bits:

$$\text{score}(g) = \alpha\,\underbrace{H(g)}_{\text{bits gained}} \;-\; \lambda\,\underbrace{\log_2 \max_f|f|}_{\text{bits left, worst case}} \;-\; \nu\,\underbrace{\log_2 \mathbb{E}[\text{rem}]}_{\text{bits left, expected}} \;+\; \mu\,\underbrace{\log_2\!\left(1 + p_{\text{ans}}(g)\right)}_{\text{credit for winning now}}$$

where $p_{\text{ans}}(g) = 1/|C|$ if $g$ is itself a surviving candidate, else $0$.

This is the user-suggested formulation with the scale problem fixed: `information_gain
- λ·worst_case + μ·answer_probability`, but with `worst_case` and `answer_probability` mapped
into bits so the coefficients are meaningful.

## Weights

Defaults $\alpha{=}1.0$, $\lambda{=}0.25$, $\nu{=}0.25$, $\mu{=}1.0$ — chosen to be
*interpretable* ("weight the two tail terms at a quarter of raw information gain"), then
checked against the benchmark rather than tuned to it. Because all terms are in bits these are
honest ratios, not arbitrary numbers. Section 19 sweeps $\lambda$ so you can see the
sensitivity instead of taking the defaults on trust.

Two exact endgame rules apply to every solver and are not heuristics: with $|C| \le 2$
remaining, probing cannot beat naming a candidate; and on the final turn a non-candidate guess
is a guaranteed loss.

In [ ]:
# Decompose the hybrid score for the opening move, to show the terms' scales.
pool_i = np.array([VOCAB.guess_index[w] for w in probe], dtype=np.int32)
s = FB.partition_stats(pool_i, all_ans)
H = s["entropy"]
worst_bits = np.log2(np.maximum(s["worst_case"], 1))
exp_bits = np.log2(np.maximum(s["expected_remaining"], 1.0))
is_c = np.array([w in VOCAB.answer_index for w in probe])
bonus = np.log2(1 + is_c / VOCAB.n_answers)
score = (ENTROPY_WEIGHT*H - MINIMAX_WEIGHT*worst_bits
         - EXPECTED_WEIGHT*exp_bits + ANSWER_BONUS*bonus)

print("hybrid score decomposition for the opening move (all terms in BITS)\n")
print(f"{'word':<8}{'a*H':>8}{'-l*worst':>10}{'-n*E[rem]':>11}{'+m*ans':>9}{'TOTAL':>9}")
print("-" * 55)
for i, w in enumerate(probe):
    print(f"{w:<8}{ENTROPY_WEIGHT*H[i]:>8.3f}{-MINIMAX_WEIGHT*worst_bits[i]:>10.3f}"
          f"{-EXPECTED_WEIGHT*exp_bits[i]:>11.3f}{ANSWER_BONUS*bonus[i]:>9.5f}"
          f"{score[i]:>9.3f}")
print("\nall four terms sit on the same bits scale, so the weights are real ratios.")

_ = demo("hybrid")

---
# 16. Full Benchmark

Every solver is evaluated against **every** answer in the discovered answer list — not a
sample. With `GUESS_POOL = "full"` the information-theoretic solvers score all 12,972 legal
guesses at every turn, which is the strongest and most honest configuration.

Reported per solver: mean / median / std / min / max guesses, the full 1–6 distribution,
failure rate, and average surviving candidates after each guess.

**Two means are reported deliberately.** `mean_guesses_solved_only` counts solved games only
(the figure normally quoted for Wordle solvers); `mean_score_failures_penalized` counts a
failure as 7. When failure rates differ between solvers, only the second is a fair single-number
comparison.

Stochastic solvers use the seed from Section 3, so runs are reproducible.

In [ ]:
answers_to_eval = VOCAB.answers[:BENCHMARK_LIMIT] if BENCHMARK_LIMIT else VOCAB.answers
print(f"evaluating {len(answers_to_eval)} answers x {len(BENCHMARK_STRATEGIES)} strategies")
print(f"guess_pool={CONFIG.guess_pool}  max_guesses={CONFIG.max_guesses}  seed={SEED}")
print("the information-theoretic solvers take a few minutes each at full pool.\n")

grand_t0 = time.perf_counter()
for strat in BENCHMARK_STRATEGIES:
    print(f"=== {strat} ===")
    res = run_benchmark(strat, FB, CONFIG, answers=answers_to_eval,
                        model=FREQ_MODEL, progress_every=500)
    RESULTS[strat] = res
    SUMMARIES[strat] = summarize(res, max_guesses=CONFIG.max_guesses)
    TRAJECTORIES[strat] = candidate_trajectory(res, max_turns=CONFIG.max_guesses)
    s = SUMMARIES[strat]
    print(f"  mean={s['mean_guesses_solved_only']:.4f}  "
          f"fail={s['failure_rate_pct']:.2f}%  wall={res.wall_s:.1f}s\n")
print(f"total {time.perf_counter()-grand_t0:.1f}s")

In [ ]:
ordered = [SUMMARIES[s] for s in BENCHMARK_STRATEGIES if s in SUMMARIES]
print(f"FULL BENCHMARK — {len(answers_to_eval)} answers, "
      f"guess_pool={CONFIG.guess_pool}, seed={SEED}")
print("=" * 104)
print(summary_table(ordered, max_guesses=CONFIG.max_guesses))
print("\n'mean' counts solved games only. Failure-penalized means:")
for s in ordered:
    print(f"  {s['strategy']:<11} solved-only={s['mean_guesses_solved_only']:.4f}   "
          f"failure-penalized={s['mean_score_failures_penalized']:.4f}   "
          f"fail={s['failure_rate_pct']:.2f}%   "
          f"({s['n_failed']} of {s['n_games']})")

In [ ]:
# Average surviving candidates after each guess.
print("AVERAGE SURVIVING CANDIDATES AFTER EACH GUESS")
print("(only games still in progress at a turn contribute to that turn)\n")
hdr = f"{'strategy':<11}" + "".join(f"{'guess '+str(t):>13}" for t in range(1, CONFIG.max_guesses+1))
print(hdr); print("-" * len(hdr))
for strat in BENCHMARK_STRATEGIES:
    tr = TRAJECTORIES.get(strat, {})
    row = f"{strat:<11}"
    for t in range(1, CONFIG.max_guesses + 1):
        row += f"{tr[t]['mean_remaining']:>13.2f}" if t in tr else f"{'-':>13}"
    print(row)

print("\nsame thing in BITS of remaining uncertainty (log2 of the candidate count)")
print(f"start: {np.log2(VOCAB.n_answers):.2f} bits\n")
print(hdr); print("-" * len(hdr))
for strat in BENCHMARK_STRATEGIES:
    tr = TRAJECTORIES.get(strat, {})
    row = f"{strat:<11}"
    for t in range(1, CONFIG.max_guesses + 1):
        row += f"{tr[t]['mean_bits_remaining']:>13.3f}" if t in tr else f"{'-':>13}"
    print(row)

In [ ]:
# Distribution histogram.
try:
    import matplotlib.pyplot as plt
    strats = [s for s in BENCHMARK_STRATEGIES if s in SUMMARIES]
    ks = list(range(1, CONFIG.max_guesses + 1))
    x = np.arange(len(ks)); w = 0.8 / max(len(strats), 1)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for i, s in enumerate(strats):
        vals = [SUMMARIES[s]["distribution_pct"].get(k, 0.0) for k in ks]
        axes[0].bar(x + i*w - 0.4 + w/2, vals, w, label=s)
    axes[0].set_xticks(x); axes[0].set_xticklabels(ks)
    axes[0].set_xlabel("guesses used"); axes[0].set_ylabel("% of games")
    axes[0].set_title(f"Guess distribution ({len(answers_to_eval)} answers)")
    axes[0].legend(fontsize=8); axes[0].grid(axis="y", alpha=.3)

    for s in strats:
        tr = TRAJECTORIES[s]
        ts = sorted(tr); axes[1].plot(ts, [tr[t]["mean_bits_remaining"] for t in ts],
                                      marker="o", label=s)
    axes[1].axhline(np.log2(VOCAB.n_answers), ls="--", c="grey",
                    label=f"start {np.log2(VOCAB.n_answers):.1f} bits")
    axes[1].set_xlabel("after guess #"); axes[1].set_ylabel("mean bits remaining")
    axes[1].set_title("Uncertainty reduction"); axes[1].legend(fontsize=8)
    axes[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib unavailable — text distribution instead:")
    for s in BENCHMARK_STRATEGIES:
        if s not in SUMMARIES: continue
        d = SUMMARIES[s]["distribution_pct"]
        print(f"\n{s}")
        for k in range(1, CONFIG.max_guesses+1):
            print(f"  {k}: {d.get(k,0):5.2f}% {'#'*int(d.get(k,0)/2)}")

---
# 17. First-Guess Analysis

Turn 1 is the only decision where every strategy faces an identical, fully known state — all
answers are possible and nothing has been observed. That makes it the cleanest place to see
what each objective actually optimises and where the objectives disagree.

All 12,972 legal guesses are scored against all 2,315 answers on entropy, expected remaining,
and worst-case bucket, with a flag for whether the word is itself a possible answer.

In [ ]:
t0 = time.perf_counter()
FGA = first_guess_analysis(FB, model=FREQ_MODEL)
print(f"scored all {VOCAB.n_guesses} openers in {time.perf_counter()-t0:.1f}s\n")

BLOCKS = [
    ("entropy",            True,  "TOP 20 BY ENTROPY  (max expected information gain, bits)"),
    ("expected_remaining", False, "TOP 20 BY EXPECTED REMAINING CANDIDATES  (minimize)"),
    ("worst_case",         False, "TOP 20 BY WORST-CASE BUCKET  (minimax)"),
    ("frequency_score",    True,  "TOP 20 BY LETTER-FREQUENCY HEURISTIC  (Layer 3)"),
]
FGA_TOP = {}
for metric, mx, title in BLOCKS:
    rows = top_openers(FGA, metric, 20, maximize=mx)
    FGA_TOP[metric] = rows
    print(format_openers(rows, "\n" + title))

In [ ]:
# What each solver actually opens with, and why.
print("OPENER CHOSEN BY EACH STRATEGY\n")
print(f"{'strategy':<11}{'opener':<8}{'H(bits)':>9}{'E[rem]':>10}{'worst':>7}"
      f"{'answer?':>9}{'H rank':>8}")
print("-" * 62)
ent_rank = {str(FGA['word'][i]): r for r, i in
            enumerate(np.argsort(-FGA["entropy"], kind="stable"), 1)}
for strat in BENCHMARK_STRATEGIES:
    op = SUMMARIES.get(strat, {}).get("opening_guess")
    if not op:
        sv = make_solver(strat, FB, CONFIG, FREQ_MODEL)
        op = sv.opening_guess() if sv.deterministic else "(random)"
    if op == "(random)":
        print(f"{strat:<11}{op:<8}{'-':>9}{'-':>10}{'-':>7}{'-':>9}{'-':>8}")
        continue
    i = VOCAB.guess_index[op]
    print(f"{strat:<11}{op:<8}{FGA['entropy'][i]:>9.4f}"
          f"{FGA['expected_remaining'][i]:>10.2f}{FGA['worst_case'][i]:>7}"
          f"{('yes' if FGA['is_possible_answer'][i] else 'no'):>9}"
          f"{ent_rank[op]:>8}")

print("\nThe objectives disagree, and the disagreement is the interesting part:")
print("  - entropy maximises AVERAGE information -> spreads probability mass evenly")
print("  - expected-remaining penalises big buckets LINEARLY, not logarithmically")
print("  - minimax ignores the average entirely and optimises only the worst bucket")
print("  - frequency never considers partitions at all, only letter statistics")
print("\nBest openers are mostly NOT possible answers — a probe's job on turn 1 is to")
print("split the space, not to win. This is why restricting guesses to the answer list")
print("would weaken the solver.")

# How much worse is a candidates-only opener?
best_any = str(FGA["word"][int(np.argmax(FGA["entropy"]))])
mask = FGA["is_possible_answer"]
ent_ans = np.where(mask, FGA["entropy"], -np.inf)
best_ans = str(FGA["word"][int(np.argmax(ent_ans))])
print(f"\nbest opener overall     : {best_any.upper()}  "
      f"H={FGA['entropy'][VOCAB.guess_index[best_any]]:.4f} bits")
print(f"best opener that is also a possible answer: {best_ans.upper()}  "
      f"H={FGA['entropy'][VOCAB.guess_index[best_ans]]:.4f} bits")

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    ia = FGA["is_possible_answer"]
    axes[0].scatter(FGA["entropy"][~ia], FGA["expected_remaining"][~ia], s=3,
                    alpha=.25, label="not a possible answer")
    axes[0].scatter(FGA["entropy"][ia], FGA["expected_remaining"][ia], s=3,
                    alpha=.45, label="possible answer")
    axes[0].set_yscale("log")
    axes[0].set_xlabel("entropy (bits)"); axes[0].set_ylabel("E[remaining] (log)")
    axes[0].set_title(f"All {VOCAB.n_guesses} openers: entropy vs expected remaining")
    axes[0].legend(markerscale=4, fontsize=8); axes[0].grid(alpha=.3)

    axes[1].hist(FGA["entropy"], bins=80)
    for strat in ("entropy", "expected", "minimax", "hybrid", "frequency"):
        op = SUMMARIES.get(strat, {}).get("opening_guess")
        if op:
            axes[1].axvline(FGA["entropy"][VOCAB.guess_index[op]], lw=1.2,
                            label=f"{strat}: {op.upper()}")
    axes[1].set_xlabel("entropy (bits)"); axes[1].set_ylabel("number of words")
    axes[1].set_title("Opener entropy distribution")
    axes[1].legend(fontsize=8); axes[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib unavailable")

---
# 18. Exact-Game Inspection

`solve(answer=..., verbose=True)` prints the full trace: each guess, the feedback, and the
number of candidates remaining. This is the tool for auditing *why* the solver did what it did
— and the format that will be useful for building supervised traces for the later LLM
experiment.

In [ ]:
r = play_game(BUNDLE.solver(STRATEGY), "crane", verbose=True)

In [ ]:
# A few deliberately awkward answers: repeated letters and dense neighbourhoods.
HARD = ["mummy", "eerie", "jazzy", "fuzzy", "wight", "vaunt", "boxer", "watch", "zonal"]
HARD = [w for w in HARD if w in VOCAB.answer_index]

for strat in ["entropy", "minimax", "hybrid"]:
    sv = make_solver(strat, FB, CONFIG, FREQ_MODEL)
    op = sv.opening_guess()
    print(f"=== {strat} (opener {op.upper()}) ===")
    for a in HARD:
        g = play_game(sv, a, first_guess=op)
        trail = "  ".join(f"{gg.upper()}[{pp}]" for gg, pp in zip(g.guesses, g.patterns))
        print(f"  {a.upper()}  {g.n_guesses if g.solved else 'FAIL'}  "
              f"cands={g.candidates_remaining}\n        {trail}")
    print()

In [ ]:
# Worst cases for the default strategy — where the classical approach struggles.
best = min((s for s in SUMMARIES.values()),
           key=lambda s: s["mean_score_failures_penalized"])["strategy"]
res = RESULTS[best]
worst = sorted(res.games, key=lambda g: -g.score)[:15]
print(f"hardest answers for '{best}':\n")
for g in worst:
    print(f"  {g.answer.upper()}  {g.n_guesses if g.solved else 'FAIL'} guesses  "
          f"cands={g.candidates_remaining}")
    print(f"        {'  '.join(x.upper() for x in g.guesses)}")
print("\nThese are dense neighbourhoods where many answers differ by one letter")
print("(-ATCH, -IGHT, -OUND ...). No amount of information-theoretic probing escapes")
print("them; the constraint structure itself is the bottleneck.")

### Interactive play

To drive the real NYT game, call `interactive_play(BUNDLE)`: it proposes a guess, you type
the feedback you saw as five characters of `G`/`Y`/`B`, and it re-filters. Left commented out
because a notebook run with no stdin will hang on the prompt.

In [ ]:
# interactive_play(BUNDLE, strategy=STRATEGY)

# Non-interactive demonstration of the same loop, feeding it scripted feedback.
_target = "crane"
_cands = np.arange(VOCAB.n_answers, dtype=np.int32)
_sv = BUNDLE.solver(STRATEGY)
print(f"(simulating interactive play against a hidden {_target.upper()})\n")
for turn in range(1, CONFIG.max_guesses + 1):
    g = _sv.opening_guess() if turn == 1 else _sv.choose(_cands, turn)
    typed = feedback(g, _target)            # what a human would type
    print(f"Guess {turn}: {g.upper()}   ({len(_cands)} candidates)")
    print(f"  feedback> {typed}")
    if pattern_to_code(typed) == ALL_GREEN:
        print(f"\nSolved in {turn} guesses: {g.upper()}")
        break
    _cands = BUNDLE.fb.filter_indices(_cands, g, pattern_to_code(typed))

---
# 19. Results Comparison — Answering the Research Question

The benchmark exists to decompose Wordle performance into the three layers. Read the table
below as a ladder: each row adds one mechanism to the row above it.

In [ ]:
LAYER = {
    "random":    ("1 symbolic only",        "exact elimination, no decision rule"),
    "frequency": ("1 + 3 heuristic",        "letter statistics, no partition reasoning"),
    "entropy":   ("1 + 2 information",      "max expected information gain"),
    "expected":  ("1 + 2 information",      "min expected remaining candidates"),
    "minimax":   ("1 + 2 information",      "min worst-case bucket"),
    "hybrid":    ("1 + 2 + 3 combined",     "bits-normalized blend + answer prior"),
}
rows = [SUMMARIES[s] for s in BENCHMARK_STRATEGIES if s in SUMMARIES]
base = SUMMARIES.get("random", {}).get("mean_score_failures_penalized")

print("CONTRIBUTION BY LAYER")
print(f"{'strategy':<11}{'layers':<22}{'mean':>8}{'vs random':>11}{'fail%':>7}  mechanism")
print("-" * 104)
for s in rows:
    lay, mech = LAYER.get(s["strategy"], ("", ""))
    d = (f"{s['mean_score_failures_penalized']-base:+.4f}"
         if base is not None else "-")
    print(f"{s['strategy']:<11}{lay:<22}"
          f"{s['mean_score_failures_penalized']:>8.4f}{d:>11}"
          f"{s['failure_rate_pct']:>7.2f}  {mech}")

bestrow = min(rows, key=lambda s: s["mean_score_failures_penalized"])
print(f"\nbest overall (failure-penalized mean): {bestrow['strategy'].upper()} "
      f"at {bestrow['mean_score_failures_penalized']:.4f}")

info = [s for s in rows if s["strategy"] in ("entropy", "expected", "minimax")]
if base is not None and info:
    bi = min(info, key=lambda s: s["mean_score_failures_penalized"])
    fr = SUMMARIES.get("frequency", {}).get("mean_score_failures_penalized")
    print("\nDECOMPOSITION")
    print(f"  symbolic elimination alone (random)        {base:.4f} guesses")
    if fr: print(f"  + frequency heuristics  (Layer 3)          {fr:.4f}  "
                 f"({base-fr:+.4f})")
    print(f"  + information theory    (Layer 2, best)    "
          f"{bi['mean_score_failures_penalized']:.4f}  "
          f"({base-bi['mean_score_failures_penalized']:+.4f})")
    print(f"  + both combined         (hybrid)           "
          f"{SUMMARIES['hybrid']['mean_score_failures_penalized']:.4f}")

In [ ]:
# Sensitivity of the hybrid to lambda, on a fixed subsample — shows the defaults
# were checked rather than asserted.
rs = np.random.default_rng(SEED)
sub = [VOCAB.answers[i] for i in rs.choice(VOCAB.n_answers, 300, replace=False)]
print(f"hybrid weight sweep on a fixed {len(sub)}-answer subsample (seed {SEED})\n")
print(f"{'lambda':>7}{'nu':>7}{'mu':>7}{'mean':>9}{'fail%':>8}")
print("-" * 38)
for lam, nu, mu in [(0.0,0.0,0.0), (0.0,0.0,1.0), (0.25,0.25,1.0),
                    (0.5,0.25,1.0), (0.25,0.5,1.0), (1.0,1.0,1.0)]:
    c = SolverConfig(max_guesses=MAX_GUESSES, seed=SEED, guess_pool=GUESS_POOL,
                     minimax_weight=lam, expected_weight=nu, answer_bonus_weight=mu)
    r = run_benchmark("hybrid", FB, c, answers=sub, model=FREQ_MODEL, log=None)
    sm = summarize(r, max_guesses=MAX_GUESSES)
    print(f"{lam:>7.2f}{nu:>7.2f}{mu:>7.2f}"
          f"{sm['mean_score_failures_penalized']:>9.4f}{sm['failure_rate_pct']:>8.2f}")
print("\n(subsample, so differences of a few thousandths are noise)")

### What this establishes for the LLM comparison

The benchmark gives the classical reference points a learned model must be measured against.
When interpreting a 0.5B LLM's Wordle scores later, three things from this notebook matter:

1. **A ceiling to beat, not a floor.** These solvers have *perfect* recall of a 12,972-word
   dictionary and do *exact* posterior filtering over 2,315 hypotheses. An LLM doing next-token
   prediction has neither for free. Matching the `random` baseline already implies the model
   learned to track constraints; matching `entropy` implies something much stronger.

2. **The layers are separable, so credit is attributable.** If a trained model beats
   `frequency` but not `entropy`, it has likely learned letter statistics rather than
   information-seeking behaviour. The gap between those two rows is the diagnostic.

3. **The honest upper bound is not here.** All of these solvers are **one-step greedy**. A full
   game-tree search over guess sequences does better than any row in this table. This baseline
   is "strong classical", not "optimal".

One caveat worth stating plainly: the solvers get the answer list as *input*, so they never
have to know which strings are words. That is a genuine advantage over an LLM, and it should
be acknowledged rather than treated as a fair fight.

---
# 20. Save Portable Artifacts

Final bundle, sufficient to reproduce the solver locally with only Python + NumPy.

In [ ]:
paths = save_artifacts(
    ARTIFACT_DIR, VOCAB, FB, FREQ_MODEL, CONFIG,
    extra_metadata={
        "matrix_build_seconds": round(BUILD_SECONDS, 2),
        "matrix_build_chunk": MATRIX_CHUNK,
        "discovered_answers_path": ANSWERS_PATH,
        "discovered_guesses_path": GUESSES_PATH,
        "guesses_file_holds_extras_only": bool(GUESSES_ARE_EXTRA),
        "vocabulary_verdict": verdict,
        "audit": {k: {kk: vv for kk, vv in v.items() if kk != "duplicate_examples"}
                  for k, v in AUDIT.items()},
        "benchmark_n_answers": len(answers_to_eval),
        "benchmark_best_strategy": bestrow["strategy"],
    },
)

# Benchmark results + full opener ranking alongside the solver bundle.
save_benchmark(os.path.join(ARTIFACT_DIR, "benchmark_results.json"),
               [SUMMARIES[s] for s in BENCHMARK_STRATEGIES if s in SUMMARIES],
               TRAJECTORIES,
               extra={"first_guess_top20": FGA_TOP,
                      "n_answers_evaluated": len(answers_to_eval),
                      "config": CONFIG.to_dict()})

order = np.argsort(-FGA["entropy"], kind="stable")
with open(os.path.join(ARTIFACT_DIR, "first_guess_analysis.csv"), "w",
          encoding="utf-8", newline="\n") as fh:
    fh.write("word,entropy_bits,expected_remaining,worst_case_remaining,"
             "is_possible_answer,frequency_score\n")
    for i in order:
        fh.write(f"{FGA['word'][i]},{FGA['entropy'][i]:.6f},"
                 f"{FGA['expected_remaining'][i]:.4f},{FGA['worst_case'][i]},"
                 f"{int(FGA['is_possible_answer'][i])},{FGA['frequency_score'][i]:.6f}\n")

# Ship the module itself so the artifact directory is self-sufficient.
import shutil
for f in ("wordle_solver.py", "benchmark.py"):
    if os.path.exists(f):
        shutil.copy(f, os.path.join(ARTIFACT_DIR, f))

print(f"ARTIFACT_DIR = {os.path.abspath(ARTIFACT_DIR)}\n")
tot = 0
for f in sorted(os.listdir(ARTIFACT_DIR)):
    sz = os.path.getsize(os.path.join(ARTIFACT_DIR, f)); tot += sz
    print(f"  {f:<30}{sz/1024:>12.1f} KiB")
print(f"  {'TOTAL':<30}{tot/1024:>12.1f} KiB")

In [ ]:
# Final gate: a fresh load of the bundle must reproduce a known game exactly.
fresh = load_artifacts(ARTIFACT_DIR, mmap=True)
chk = play_game(fresh.solver(STRATEGY), "crane")
print(f"fresh bundle solves CRANE in {chk.n_guesses}: "
      f"{' -> '.join(g.upper() for g in chk.guesses)}")
assert chk.solved
print("\nartifacts are complete and self-sufficient.")

---
# 21. Local Usage Instructions

## Download

On Kaggle, take everything from `/kaggle/working/artifacts/`. You need at minimum:

```
artifacts/
    answers.txt                 2,315 answers, one per line
    valid_guesses.txt          12,972 legal guesses, one per line
    feedback_matrix.npy        12,972 x 2,315 uint8  (~28.6 MiB)
    metadata.json              shapes, encoding, provenance, audit
    frequency_model.json       letter / positional statistics
    solver_config.json         seed, weights, guess-pool policy
    wordle_solver.py           the solver module itself
    benchmark.py               the evaluation harness
```

Plus, for reference: `benchmark_results.json` and `first_guess_analysis.csv`.

## Install

Only NumPy is required.

```bash
conda create -n wordle python=3.11 numpy
conda activate wordle
```

## Run

```bash
python wordle_solver.py --info
python wordle_solver.py --answer crane
python wordle_solver.py --answer crane --strategy minimax
python wordle_solver.py --interactive          # play the real NYT game
```

From Python:

```python
from wordle_solver import load_artifacts, play_game, solve

bundle = load_artifacts("artifacts")     # instant, memory-mapped
solve(answer="crane", verbose=True)

solver = bundle.solver("entropy")
r = play_game(solver, "mummy")
print(r.n_guesses, r.guesses, r.candidates_remaining)
```

Rebuilding from scratch (no artifacts needed, ~50 s):

```bash
python build_artifacts.py --data-dir data --artifact-dir artifacts
python run_full_benchmark.py
pytest tests/ -q
```

## Changing behaviour

Everything lives in `SolverConfig` (`solver_config.json`, or pass it directly):

```python
from wordle_solver import SolverConfig, load_artifacts, make_solver

cfg = SolverConfig(
    seed=20260817, max_guesses=6, strategy="hybrid",
    guess_pool="full",            # "full" | "adaptive" | "candidates"
    entropy_weight=1.0, minimax_weight=0.25,
    expected_weight=0.25, answer_bonus_weight=1.0,
    opening_guess="soare",        # skip recomputing turn 1
)
b = load_artifacts("artifacts")
solver = make_solver("hybrid", b.fb, cfg, b.model)
```

Set `guess_pool="adaptive"` for a large speedup at a very small cost in mean guesses.

## Portability notes

- Paths are relative; `DATA_DIR` / `ARTIFACT_DIR` are the only path knobs.
- No `/kaggle/...` path appears anywhere in `wordle_solver.py` — Kaggle detection lives only
  in Section 2 of the notebook.
- `.npy` is portable across Windows and Linux; the matrix is `uint8`, so endianness is moot.
- Dependencies: Python ≥ 3.9 and NumPy. `pandas`/`matplotlib` are used only for notebook
  visualisation and are optional. No ML, LLM, or RL libraries anywhere.
- CPU only. A GPU would give no benefit at this problem size, so none is used.